# Notebook 05 — Feature Detection & Matching (Classical + Learned)

**Vision & 3D Mapping Workshop** | Block 3: Multi-View Geometry

---

## Why This Matters

Multi-view geometry begins with a deceptively simple question: *given two photographs of the
same scene, which pixel in image A corresponds to which pixel in image B?*  The answer
requires **feature detection** (finding interesting points), **feature description** (encoding
their local appearance), and **feature matching** (establishing correspondences).

These correspondences are the raw material from which we estimate the **fundamental matrix**,
the **essential matrix**, and ultimately the **3-D structure** of the scene.  A single bad
match can corrupt the entire reconstruction, so robust estimation methods like **RANSAC**
are indispensable.

### What You'll Learn

1. **Harris corner detector** — structure tensor, corner response, eigenvalue analysis
2. **FAST detector** — Bresenham circle test, segment test, decision-tree speedup
3. **ORB** — oriented FAST + rotated BRIEF, Hamming distance matching
4. **SIFT** — scale-space, DoG, orientation assignment, 128-dim descriptor
5. **Feature matching** — brute-force, FLANN, Lowe's ratio test derivation
6. **RANSAC** — robust estimation with outlier rejection, iteration bound derivation
7. **Fundamental matrix** — 8-point algorithm, Hartley normalization, rank-2 enforcement
8. **Essential matrix** — decomposition into $(R,t)$, cheirality check
9. **Learned features** — SuperPoint, LightGlue, LoFTR (with attention math), Efficient LoFTR, RoMa/RoMa v2, OmniGlue, LoMa
10. **Matching taxonomy** — sparse vs. semi-dense vs. dense paradigms, tradeoffs
11. **Connection to 3-D** — how DUSt3R/MASt3R implicitly perform dense matching
12. **Exercises** — guided implementations with skeleton code

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.signal import convolve2d

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "image.cmap": "gray",
    "axes.grid": False,
})

---
## Synthetic Data Utilities

We create helper functions that generate all test images used throughout the notebook.
Everything is self-contained — no external datasets required.

In [ ]:
def make_checkerboard(rows=8, cols=8, square_size=40, noise_std=5.0):
    """Generate a checkerboard image with optional Gaussian noise."""
    h, w = rows * square_size, cols * square_size
    img = np.zeros((h, w), dtype=np.float64)
    for i in range(rows):
        for j in range(cols):
            if (i + j) % 2 == 0:
                r0, r1 = i * square_size, (i + 1) * square_size
                c0, c1 = j * square_size, (j + 1) * square_size
                img[r0:r1, c0:c1] = 255.0
    if noise_std > 0:
        img += np.random.randn(h, w) * noise_std
    return np.clip(img, 0, 255).astype(np.uint8)


def make_shapes_image(h=300, w=400):
    """Generate an image with random geometric shapes."""
    img = np.zeros((h, w), dtype=np.uint8)
    rng = np.random.RandomState(42)
    for _ in range(5):
        cx, cy = rng.randint(50, w - 50), rng.randint(50, h - 50)
        r = rng.randint(20, 60)
        cv2.circle(img, (cx, cy), r, int(rng.randint(100, 255)), -1)
    for _ in range(5):
        x1, y1 = rng.randint(10, w - 10), rng.randint(10, h - 10)
        x2, y2 = x1 + rng.randint(20, 80), y1 + rng.randint(20, 80)
        cv2.rectangle(img, (x1, y1), (x2, y2), int(rng.randint(100, 255)), -1)
    for _ in range(6):
        pts = rng.randint(10, min(h, w) - 10, size=(3, 2))
        cv2.fillPoly(img, [pts], int(rng.randint(100, 255)))
    return img


def make_two_view_scene(n_points=80, noise_px=0.5):
    """
    Generate a synthetic two-view scenario:
      - 3D points in front of both cameras
      - Two camera intrinsic matrices (identical)
      - Known R, t between cameras
      - 2D projections in both views with optional noise
    Returns: pts3d, K, R, t, pts1, pts2
    """
    rng = np.random.RandomState(42)
    pts3d = rng.randn(n_points, 3) * np.array([2.0, 2.0, 0.5]) + np.array([0, 0, 5])

    K = np.array([[500, 0, 320],
                  [0, 500, 240],
                  [0,   0,   1]], dtype=np.float64)

    angle = np.deg2rad(8)
    R = np.array([[ np.cos(angle), 0, np.sin(angle)],
                  [ 0,             1, 0            ],
                  [-np.sin(angle), 0, np.cos(angle)]], dtype=np.float64)
    t = np.array([[0.5], [-0.1], [0.05]], dtype=np.float64)

    P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
    P2 = K @ np.hstack([R, t])

    pts3d_h = np.hstack([pts3d, np.ones((n_points, 1))]).T  # 4 x N
    proj1 = P1 @ pts3d_h  # 3 x N
    proj2 = P2 @ pts3d_h

    pts1 = (proj1[:2] / proj1[2:]).T  # N x 2
    pts2 = (proj2[:2] / proj2[2:]).T

    if noise_px > 0:
        pts1 += rng.randn(*pts1.shape) * noise_px
        pts2 += rng.randn(*pts2.shape) * noise_px

    return pts3d, K, R, t, pts1, pts2

In [ ]:
checker = make_checkerboard()
shapes = make_shapes_image()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(checker, cmap="gray")
axes[0].set_title("Synthetic Checkerboard")
axes[1].imshow(shapes, cmap="gray")
axes[1].set_title("Synthetic Shapes")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## 1. The Structure Tensor & Harris Corner Detector

Corner detection is the entry point to virtually all geometric vision in autonomous systems.
A drone's visual odometry tracks Harris or FAST corners between frames to estimate its
ego-motion; ORB-SLAM builds its entire map from ORB features (which are FAST corners +
BRIEF descriptors); and even learned matchers like SuperPoint detect corner-like keypoints.
Understanding *why* corners are special — and *when* detection fails — is essential for
building reliable perception.

### 1.1 What Makes a Corner?

Intuitively, a **corner** is a point where the image intensity changes significantly in
*multiple* directions.  Contrast this with an **edge** (intensity changes in one direction
only) and a **flat region** (no change in any direction).

#### Historical Development

The idea of detecting corners via local auto-correlation has a rich lineage:

1. **Moravec (1980)** — first corner detector for robot navigation; compared SSD over
   discrete shifts (0°, 45°, 90°, 135°). Limited by anisotropy of the discrete shifts.
2. **Förstner & Gülch (1987)** — introduced the structure tensor $M$ and used the
   *inverse* of $M$ to define an error ellipse; the smallest eigenvalue of $M^{-1}$
   gave the "point distinctness" criterion. First rigorous formulation.
3. **Harris & Stephens (1988)** — simplified Förstner's approach using the response
   function $R = \det(M) - k\,\text{trace}(M)^2$, avoiding explicit eigenvalue
   computation. This is what we implement below.
4. **Shi & Tomasi (1994)** — proved that the criterion $R_{\text{ST}} = \min(\lambda_1, \lambda_2)$
   (i.e., *the smallest eigenvalue alone*) gives better features for Lucas-Kanade tracking
   than the Harris response. Their paper title — "Good Features to Track" — became
   a standard term.

Harris & Stephens formalized the Moravec intuition using the **auto-correlation function**:
shift a small window by $(u, v)$ and measure the sum of squared differences (SSD).

$$
E(u, v) = \sum_{(x,y) \in W} w(x,y)\, \bigl[ I(x+u,\, y+v) - I(x,y) \bigr]^2
$$

where $w(x,y)$ is a Gaussian weighting window.  A first-order Taylor expansion gives

$$
I(x+u,\, y+v) \approx I(x,y) + I_x u + I_y v
$$

so

$$
E(u, v) \approx \sum_{(x,y) \in W} w(x,y)\, (I_x u + I_y v)^2
= \begin{bmatrix} u & v \end{bmatrix}
\underbrace{\left( \sum_{(x,y) \in W} w(x,y) \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix} \right)}_{M}
\begin{bmatrix} u \\ v \end{bmatrix}.
$$

### 1.2 The Structure Tensor

> **Notation:** The matrix **M** (sometimes called **H** in the Harris literature) is the structure tensor / second-moment matrix. We use **M** to avoid confusion with the homography matrix **H** used in NB02.

The $2 \times 2$ matrix $M$ is called the **structure tensor** (or second-moment matrix):

$$
M = G_\sigma * \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}
$$

where $G_\sigma *$ denotes convolution with a Gaussian of standard deviation $\sigma$
(replacing the discrete window summation with a smooth weighting).

### 1.3 Image Gradients via Sobel Operators

The partial derivatives $I_x, I_y$ are approximated by convolving with Sobel kernels:

$$
S_x = \begin{bmatrix} -1 & 0 & +1 \\ -2 & 0 & +2 \\ -1 & 0 & +1 \end{bmatrix}, \qquad
S_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ +1 & +2 & +1 \end{bmatrix}.
$$

Each kernel is separable: $S_x = \begin{bmatrix}1\\2\\1\end{bmatrix} \begin{bmatrix}-1&0&1\end{bmatrix}$,
combining a smoothing (binomial) filter in one direction with a central-difference derivative
in the other.

### 1.4 Eigenvalue Analysis

Since $M$ is real and symmetric, it has two real eigenvalues $\lambda_1 \geq \lambda_2 \geq 0$.
The SSD function $E(u,v)$ is a quadratic form whose principal axes are the eigenvectors of $M$,
with lengths proportional to the eigenvalues.

| Region | Eigenvalues | Interpretation |
|:---|:---|:---|
| **Flat** | $\lambda_1 \approx \lambda_2 \approx 0$ | No intensity change in any direction |
| **Edge** | $\lambda_1 \gg \lambda_2 \approx 0$ | Strong change in one direction only |
| **Corner** | $\lambda_1, \lambda_2$ both large | Strong change in two independent directions |

### 1.5 Corner Response Functions

Computing eigenvalues explicitly requires solving a $2 \times 2$ eigenvalue problem per pixel.
Several scalar **response functions** avoid this by combining matrix invariants:

| Response | Formula | Reference |
|----------|---------|-----------|
| **Harris** | $R = \det(M) - k\,\text{trace}(M)^2$ | Harris & Stephens, 1988 |
| **Shi-Tomasi** | $R = \min(\lambda_1, \lambda_2)$ | Shi & Tomasi, 1994 |
| **Noble** | $R = \frac{\det(M)}{\text{trace}(M) + \varepsilon}$ | Noble, 1988 |
| **Harmonic mean** | $R = \frac{\det(M)}{\text{trace}(M)}$ | Förstner & Gülch, 1987 |

Harris is the most widely used because it avoids eigenvalue computation entirely:

$$
R = \det(M) - k \cdot \text{trace}(M)^2
$$

where $k \in [0.04, 0.06]$ is an empirical constant (typically $k = 0.04$).

**Why does this work?** Recall the fundamental relations between eigenvalues and matrix
invariants:

$$
\det(M) = \lambda_1 \lambda_2, \qquad \text{trace}(M) = \lambda_1 + \lambda_2.
$$

*Proof:* $M$ is similar to $\text{diag}(\lambda_1, \lambda_2)$, so they share the same
determinant and trace. For a diagonal matrix, $\det = \lambda_1 \lambda_2$ and
$\text{trace} = \lambda_1 + \lambda_2$. $\square$

Substituting:

$$
R = \lambda_1 \lambda_2 - k (\lambda_1 + \lambda_2)^2.
$$

Analyzing the cases:

- **Corner** ($\lambda_1, \lambda_2$ both large): $\lambda_1 \lambda_2$ is large,
  $(\lambda_1+\lambda_2)^2$ is also large, but the product dominates for balanced
  eigenvalues $\Rightarrow R > 0$ (positive, large).
- **Edge** ($\lambda_1 \gg \lambda_2 \approx 0$): $\lambda_1 \lambda_2 \approx 0$ but
  $(\lambda_1 + \lambda_2)^2 \approx \lambda_1^2$ is large $\Rightarrow R < 0$ (negative).
- **Flat** ($\lambda_1 \approx \lambda_2 \approx 0$): Both terms vanish $\Rightarrow R \approx 0$.

> **Why $R > 0$ for corners (precisely):** For balanced eigenvalues $\lambda_1 \approx \lambda_2 = \lambda$:
> $R = \lambda^2 - k(2\lambda)^2 = \lambda^2(1 - 4k)$. Since $k < 0.25$ in practice
> (typically $0.04$–$0.06$), the factor $(1 - 4k)$ is always positive, guaranteeing $R > 0$
> for any corner with non-zero eigenvalues.

### 1.6 Non-Maximum Suppression (NMS)

After thresholding the response $R > R_{\min}$, we keep only **local maxima**: a pixel is
retained only if its response is the maximum in a $w \times w$ neighborhood.

### Visualizing the Eigenvalue Landscape

Before implementing Harris, let's visualize the response $R(\lambda_1, \lambda_2)$ to build
intuition about the decision boundary between corners, edges, and flat regions.

In [ ]:
lam = np.linspace(0, 10, 200)
L1, L2 = np.meshgrid(lam, lam)
k_val = 0.04
R_landscape = L1 * L2 - k_val * (L1 + L2) ** 2

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.contourf(L1, L2, R_landscape, levels=40, cmap="RdBu_r")
ax.contour(L1, L2, R_landscape, levels=[0], colors="black", linewidths=2)
plt.colorbar(im, ax=ax, label="$R = \\lambda_1 \\lambda_2 - k(\\lambda_1+\\lambda_2)^2$")
ax.set_xlabel("$\\lambda_1$")
ax.set_ylabel("$\\lambda_2$")
ax.set_title("Harris response in eigenvalue space ($k = 0.04$)")
ax.set_aspect("equal")

ax.annotate("Corner\n$R > 0$", xy=(7, 7), fontsize=12, ha="center",
            bbox=dict(boxstyle="round", fc="white", alpha=0.8))
ax.annotate("Edge\n$R < 0$", xy=(9, 0.5), fontsize=12, ha="center",
            bbox=dict(boxstyle="round", fc="white", alpha=0.8))
ax.annotate("Flat\n$R \\approx 0$", xy=(1, 1), fontsize=12, ha="center",
            bbox=dict(boxstyle="round", fc="white", alpha=0.8))

plt.tight_layout()
plt.show()

### 1.7 From-Scratch Implementation

We now implement the full Harris detector step by step.

In [ ]:
def harris_corner_detector(img, k=0.04, gauss_sigma=1.5, nms_size=7, threshold_ratio=0.01):
    """
    Harris corner detector implemented from scratch.

    Parameters
    ----------
    img : 2-D uint8 array
    k : Harris free parameter
    gauss_sigma : Gaussian smoothing sigma for the structure tensor
    nms_size : window size for non-maximum suppression
    threshold_ratio : keep corners with R > threshold_ratio * max(R)

    Returns
    -------
    corners : (N, 2) array of (row, col) corner locations
    R : response map (same shape as img)
    intermediates : dict with Ix, Iy, Ixx, Ixy, Iyy for visualization
    """
    img_f = img.astype(np.float64)

    Ix = cv2.Sobel(img_f, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(img_f, cv2.CV_64F, 0, 1, ksize=3)

    Ixx = gaussian_filter(Ix * Ix, sigma=gauss_sigma)
    Ixy = gaussian_filter(Ix * Iy, sigma=gauss_sigma)
    Iyy = gaussian_filter(Iy * Iy, sigma=gauss_sigma)

    det_M = Ixx * Iyy - Ixy * Ixy
    trace_M = Ixx + Iyy
    R = det_M - k * trace_M ** 2

    threshold = threshold_ratio * R.max()
    R_thresh = np.where(R > threshold, R, 0)

    local_max = maximum_filter(R_thresh, size=nms_size)
    corners_mask = (R_thresh == local_max) & (R_thresh > 0)
    corners = np.argwhere(corners_mask)  # (row, col)

    intermediates = {"Ix": Ix, "Iy": Iy, "Ixx": Ixx, "Ixy": Ixy, "Iyy": Iyy,
                     "det_M": det_M, "trace_M": trace_M}
    return corners, R, intermediates

In [ ]:
corners, R, inter = harris_corner_detector(checker)
print(f"Detected {len(corners)} corners on the checkerboard.")

### 1.8 Visualizing Every Stage

Below we show the intermediate outputs: image gradients, structure tensor components,
response map, and final detected corners.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

axes[0, 0].imshow(checker, cmap="gray")
axes[0, 0].set_title("Original")

axes[0, 1].imshow(inter["Ix"], cmap="RdBu_r")
axes[0, 1].set_title(r"$I_x$ (Sobel horizontal)")

axes[0, 2].imshow(inter["Iy"], cmap="RdBu_r")
axes[0, 2].set_title(r"$I_y$ (Sobel vertical)")

axes[0, 3].imshow(inter["Ixx"], cmap="hot")
axes[0, 3].set_title(r"$G_\sigma * I_x^2$")

axes[1, 0].imshow(inter["Ixy"], cmap="RdBu_r")
axes[1, 0].set_title(r"$G_\sigma * I_x I_y$")

axes[1, 1].imshow(inter["Iyy"], cmap="hot")
axes[1, 1].set_title(r"$G_\sigma * I_y^2$")

im = axes[1, 2].imshow(R, cmap="jet")
axes[1, 2].set_title("Harris response $R$")
plt.colorbar(im, ax=axes[1, 2], fraction=0.046)

axes[1, 3].imshow(checker, cmap="gray")
R_overlay = np.where(R > R.max() * 0.01, R, np.nan)
axes[1, 3].imshow(R_overlay, cmap="hot", alpha=0.5)
axes[1, 3].plot(corners[:, 1], corners[:, 0], "c+", markersize=8, markeredgewidth=1.5)
axes[1, 3].set_title(f"Corners + response overlay ({len(corners)})")

for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Harris Corner Detector — Step by Step", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 1.9 Effect of Parameter $k$

The parameter $k$ controls the sensitivity of the detector.  Larger $k$ requires eigenvalues
to be more balanced (i.e., stricter corner criterion).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, k_test in zip(axes, [0.01, 0.04, 0.06, 0.15]):
    c, _, _ = harris_corner_detector(checker, k=k_test)
    ax.imshow(checker, cmap="gray")
    ax.plot(c[:, 1], c[:, 0], "r+", markersize=6)
    ax.set_title(f"$k = {k_test}$  ({len(c)} corners)")
    ax.axis("off")
plt.suptitle("Effect of $k$ on Harris detection", fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. ORB (Oriented FAST and Rotated BRIEF)

### 3.1 Overview

**ORB** (Rublee et al., 2011) is a fast, rotation-invariant binary feature designed as a
free alternative to SIFT/SURF.  It is the default feature in **ORB-SLAM** and
**ORB-SLAM2/3**, the most widely deployed visual SLAM systems.

ORB combines:
1. **FAST keypoint detector** with Harris score filtering
2. **Intensity centroid** for orientation assignment
3. **Steered BRIEF** for rotation-invariant binary description

### 3.2 FAST Keypoints + Harris Score Filtering

ORB detects keypoints using FAST-9 on an image pyramid (for scale invariance), then
ranks them by their **Harris corner response** to reject edge-like detections.  Only the
top-$N$ keypoints (by Harris score) are retained.

The following subsections (§3.3–3.6) detail each stage of the ORB pipeline.

### 3.3 Orientation via Intensity Centroid

The orientation of each keypoint is computed from the **intensity centroid** of the patch.
Define the image moments in a circular patch of radius $r$ around the keypoint:

$$
m_{pq} = \sum_{x=-r}^{r} \sum_{y=-r}^{r} x^p \, y^q \, I(x, y), \qquad
\text{(for $x^2+y^2 \leq r^2$)}
$$

The centroid is at $(m_{10}/m_{00},\; m_{01}/m_{00})$, and the orientation is

$$
\theta = \text{atan2}(m_{01},\; m_{10}).
$$

This gives a canonical angle to rotate the descriptor, achieving **rotation invariance**.

### 3.4 BRIEF Descriptor

**BRIEF** (Binary Robust Independent Elementary Features) encodes a patch as a **binary
string** of length $n$ (typically 256).  For each bit $i$, two pixel locations
$(\mathbf{p}_i, \mathbf{q}_i)$ in the patch are compared:

$$
\tau(\mathbf{p}_i, \mathbf{q}_i) = \begin{cases} 1 & \text{if } I(\mathbf{p}_i) < I(\mathbf{q}_i) \\ 0 & \text{otherwise} \end{cases}
$$

The descriptor is the concatenation $d = \bigl(\tau_1, \tau_2, \ldots, \tau_n\bigr) \in \{0,1\}^n$.

### 3.5 Steered BRIEF

To make BRIEF rotation-invariant, ORB **rotates** the sampling pattern by the keypoint
orientation $\theta$.  If the original sampling pair is $(\mathbf{p}_i, \mathbf{q}_i)$,
the steered pair is $R_\theta \mathbf{p}_i, R_\theta \mathbf{q}_i$ where
$R_\theta = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$.

Additionally, ORB selects sampling pairs that are **decorrelated** (via a greedy search
over candidate pairs), maximizing discriminative power.

### 3.6 Hamming Distance Matching

Binary descriptors are matched using **Hamming distance**: the number of bits that differ.
This is computed extremely efficiently via XOR followed by popcount (population count):

$$
d_H(\mathbf{a}, \mathbf{b}) = \text{popcount}(\mathbf{a} \oplus \mathbf{b})
$$

On modern CPUs, popcount is a single instruction, making binary matching orders of
magnitude faster than L2 distance on float descriptors.

### 3.7 Generating Two Synthetic Views

In [ ]:
checker_big = make_checkerboard(rows=6, cols=8, square_size=50, noise_std=3.0)

rows_cb, cols_cb = checker_big.shape
M_translate = np.float32([[1, 0, 25], [0, 1, 15]])
checker_translated = cv2.warpAffine(checker_big, M_translate, (cols_cb, rows_cb))

center = (cols_cb // 2, rows_cb // 2)
M_rotate = cv2.getRotationMatrix2D(center, angle=12, scale=1.0)
checker_rotated = cv2.warpAffine(checker_big, M_rotate, (cols_cb, rows_cb))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(checker_big, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(checker_translated, cmap="gray")
axes[1].set_title("Translated")
axes[2].imshow(checker_rotated, cmap="gray")
axes[2].set_title("Rotated 12°")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3.8 ORB Detection and Matching

In [ ]:
orb = cv2.ORB_create(nfeatures=300)

kp1, des1 = orb.detectAndCompute(checker_big, None)
kp2, des2 = orb.detectAndCompute(checker_rotated, None)

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)
matches = sorted(matches, key=lambda m: m.distance)

img_matches = cv2.drawMatches(
    checker_big, kp1, checker_rotated, kp2, matches[:40], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(14, 5))
plt.imshow(img_matches)
plt.title(f"ORB: Top 40 matches (Hamming distance, cross-check) out of {len(matches)}")
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"Keypoints: view1={len(kp1)}, view2={len(kp2)}")
print(f"Matches after cross-check: {len(matches)}")
print(f"Descriptor shape: {des1.shape} (dtype: {des1.dtype})")
print(f"Descriptor length: {des1.shape[1] * 8} bits = {des1.shape[1]} bytes")

### 3.9 Hamming Distance Distribution

In [ ]:
distances = [m.distance for m in matches]
plt.figure(figsize=(8, 4))
plt.hist(distances, bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Hamming distance")
plt.ylabel("Count")
plt.title("Distribution of ORB match Hamming distances")
plt.axvline(np.median(distances), color="red", linestyle="--",
            label=f"Median = {np.median(distances):.0f}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. SIFT (Scale-Invariant Feature Transform)

### 4.1 Scale-Space Representation

A feature that is a corner at one scale might look like an edge or a flat region at another.
**Scale-space theory** (Lindeberg, 1994) defines a family of smoothed images:

$$
L(x, y, \sigma) = G(x, y, \sigma) * I(x, y)
$$

where $G(x,y,\sigma) = \frac{1}{2\pi\sigma^2} \exp\!\left(-\frac{x^2+y^2}{2\sigma^2}\right)$
is the Gaussian kernel.  The parameter $\sigma$ controls the scale: larger $\sigma$ means
coarser (more blurred) representation.

### 4.2 Difference of Gaussians (DoG)

The Laplacian of Gaussian $\nabla^2 G$ is the ideal blob detector, but SIFT approximates it
with the **Difference of Gaussians** (DoG), which is much cheaper to compute:

$$
D(x, y, \sigma) = L(x, y, k\sigma) - L(x, y, \sigma) \approx (k-1)\,\sigma^2\,\nabla^2 G * I
$$

where $k$ is the ratio between consecutive scales (typically $k = 2^{1/3}$).

#### Derivation: Why DoG $\approx \sigma^2 \nabla^2 G$

**Step 1 — Taylor expansion in scale.** Treat $G(x,y;\sigma)$ as a function of
$\sigma$ and expand $G(x,y;k\sigma)$ around $\sigma$:

$$
G(x,y;k\sigma) = G(x,y;\sigma) + (k\sigma - \sigma)\,\frac{\partial G}{\partial \sigma}
  + \mathcal{O}\bigl((k-1)^2\sigma^2\bigr)
$$

Therefore:

$$
D = G(x,y;k\sigma) - G(x,y;\sigma) \approx (k-1)\,\sigma\,\frac{\partial G}{\partial \sigma}
$$

**Step 2 — The heat-equation identity.** The Gaussian kernel satisfies the
**diffusion (heat) equation** $\partial G / \partial t = \tfrac{1}{2}\nabla^2 G$
with $t = \sigma^2$.  Applying the chain rule $\partial G/\partial \sigma
= (\partial G / \partial t)(\partial t / \partial \sigma) = \sigma\,\nabla^2 G$
gives the identity:

$$
\frac{\partial G}{\partial \sigma} = \sigma\,\nabla^2 G
$$

**Step 3 — Combining.** Substituting into the Taylor result:

$$
D \;\approx\; (k-1)\,\sigma \cdot \sigma\,\nabla^2 G
   \;=\; (k-1)\,\sigma^2\,\nabla^2 G
$$

Convolving both sides with the image $I$ (and using commutativity
$\nabla^2 G * I = \nabla^2(G * I) = \nabla^2 L$):

$$
D * I \;\approx\; (k-1)\,\sigma^2\,\nabla^2 L
$$

The approximation improves as $k \to 1$ (smaller spacing between scales). More precisely,
$D/(k-1) \to \sigma^2 \nabla^2 L$ as $k \to 1$, where $L = G_\sigma * I$ is the
smoothed image.

### 4.3 Keypoint Localization

Keypoints are detected as **local extrema** in the 3-D space $(x, y, \sigma)$: a point must
be the minimum or maximum among its 26 neighbors ($8 + 9 + 9$ in the current, upper, and
lower DoG levels).

Sub-pixel refinement uses a **Taylor expansion** of $D(x, y, \sigma)$ around the detected
location $\mathbf{x}_0$:

$$
D(\mathbf{x}) \approx D(\mathbf{x}_0) + \frac{\partial D}{\partial \mathbf{x}}^\top \Delta\mathbf{x}
+ \frac{1}{2} \Delta\mathbf{x}^\top \frac{\partial^2 D}{\partial \mathbf{x}^2} \Delta\mathbf{x}
$$

Setting the gradient to zero gives the sub-pixel offset:

$$
\Delta\mathbf{x} = -\left(\frac{\partial^2 D}{\partial \mathbf{x}^2}\right)^{-1} \frac{\partial D}{\partial \mathbf{x}}
$$

Keypoints with low contrast ($|D(\mathbf{x}_0 + \Delta\mathbf{x})|$ below threshold) or that
lie on edges (detected via ratio of principal curvatures from the Hessian, similar to Harris)
are rejected.

### 4.4 Orientation Assignment

For each keypoint at scale $\sigma$, gradient magnitudes and orientations are computed in a
neighborhood:

$$
m(x,y) = \sqrt{(L(x+1,y) - L(x-1,y))^2 + (L(x,y+1) - L(x,y-1))^2}
$$

$$
\theta(x,y) = \text{atan2}(L(x,y+1) - L(x,y-1),\; L(x+1,y) - L(x-1,y))
$$

A **36-bin orientation histogram** (10° per bin) is accumulated, weighted by $m$ and a
Gaussian window.  The dominant peak (and any peak above 80% of the maximum) defines the
keypoint orientation(s).  Parabolic interpolation refines the peak location.

### 4.5 SIFT Descriptor Construction

The descriptor is built from the gradient field in a $16 \times 16$ patch around the
keypoint, rotated to align with the keypoint orientation:

1. Divide the patch into a $4 \times 4$ grid of sub-regions.
2. In each sub-region, accumulate an **8-bin orientation histogram** of gradient magnitudes
   (weighted by a Gaussian centered on the keypoint).
3. Concatenate: $4 \times 4 \times 8 = 128$ floating-point values.
4. **Normalize** to unit length, clamp values > 0.2, re-normalize.  This reduces the
   influence of large gradients (illumination changes).

The resulting 128-dimensional float vector is the SIFT descriptor.

In [ ]:
sift = cv2.SIFT_create()

shapes_for_sift = make_shapes_image(h=400, w=600)
kps_sift, des_sift = sift.detectAndCompute(shapes_for_sift, None)

img_kps = cv2.drawKeypoints(
    shapes_for_sift, kps_sift, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

plt.figure(figsize=(12, 5))
plt.imshow(img_kps)
plt.title(f"SIFT keypoints with scale & orientation ({len(kps_sift)})")
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"Descriptor shape: {des_sift.shape}")
print(f"Descriptor dtype: {des_sift.dtype}")

### 4.6 Inspecting a SIFT Descriptor

In [ ]:
if des_sift is not None and len(des_sift) > 0:
    desc_idx = 0
    desc = des_sift[desc_idx]
    desc_grid = desc.reshape(4, 4, 8)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    for i in range(4):
        for j in range(4):
            offset = (i * 4 + j) * 8
            ax.bar(np.arange(8) + offset, desc_grid[i, j], width=0.8, alpha=0.7)
    ax.set_xlabel("Bin index (4×4 cells × 8 bins = 128)")
    ax.set_ylabel("Magnitude")
    ax.set_title(f"SIFT descriptor for keypoint {desc_idx}")

    ax2 = axes[1]
    ax2.imshow(desc_grid.sum(axis=2), cmap="hot", interpolation="nearest")
    ax2.set_title("Energy per 4×4 cell")
    ax2.set_xlabel("Cell column")
    ax2.set_ylabel("Cell row")
    plt.colorbar(ax2.images[0], ax=ax2, fraction=0.046)

    plt.tight_layout()
    plt.show()

### 4.7 Visualizing the Scale-Space and DoG Pyramid

In [ ]:
img_ss = shapes_for_sift.astype(np.float64)
sigmas = [1.0, 1.6, 2.0, 2.5, 3.2, 4.0]

scale_space = [gaussian_filter(img_ss, s) for s in sigmas]
dogs = [scale_space[i + 1] - scale_space[i] for i in range(len(sigmas) - 1)]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, (L, s) in enumerate(zip(scale_space[:5], sigmas[:5])):
    axes[0, i].imshow(L, cmap="gray")
    axes[0, i].set_title(f"$L(x,y,{s:.1f})$")
    axes[0, i].axis("off")

for i, D in enumerate(dogs[:5]):
    axes[1, i].imshow(D, cmap="RdBu_r")
    axes[1, i].set_title(f"DoG {i}")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Scale-space $L$", fontsize=12)
axes[1, 0].set_ylabel("DoG $D$", fontsize=12)
plt.suptitle("Scale-Space and Difference-of-Gaussians Pyramid", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 4.8 SIFT vs. ORB — Side-by-Side Comparison

In [ ]:
orb_cmp = cv2.ORB_create(nfeatures=500)
kps_orb_cmp, _ = orb_cmp.detectAndCompute(shapes_for_sift, None)

img_sift_vis = cv2.drawKeypoints(shapes_for_sift, kps_sift, None,
                                  flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
img_orb_vis = cv2.drawKeypoints(shapes_for_sift, kps_orb_cmp, None,
                                 flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img_sift_vis)
axes[0].set_title(f"SIFT ({len(kps_sift)} keypoints)")
axes[1].imshow(img_orb_vis)
axes[1].set_title(f"ORB ({len(kps_orb_cmp)} keypoints)")
for ax in axes:
    ax.axis("off")
plt.suptitle("SIFT vs ORB keypoints on same image", fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Feature Matching

### 5.1 Brute-Force Matching

Given descriptors $\{\mathbf{d}_i\}_{i=1}^{N_1}$ from image 1 and
$\{\mathbf{d}_j\}_{j=1}^{N_2}$ from image 2, the **brute-force** matcher computes
all $N_1 \times N_2$ pairwise distances and finds the nearest neighbor for each
query descriptor:

$$
j^* = \arg\min_{j} \| \mathbf{d}_i - \mathbf{d}_j \|_2
$$

For SIFT (float descriptors), L2 distance is standard.  For ORB (binary),
Hamming distance is used.

### 5.2 FLANN (Fast Library for Approximate Nearest Neighbors)

Brute-force is $O(N_1 N_2 D)$ where $D$ is the descriptor dimension — too slow for
large feature sets.  **FLANN** uses spatial data structures for sub-linear search:

- **kd-trees**: Partition the descriptor space along coordinate axes.  Search backtracks
  through the tree, pruning branches whose closest point is farther than the current best.
  Effective for $D \lesssim 20$; for SIFT ($D=128$), multiple randomized kd-trees are used.
- **Randomized kd-trees**: Build several trees with random axis selection, search them
  in parallel. Empirically much faster than a single tree in high dimensions.
- **Hierarchical k-means tree**: Recursively partition the data using k-means. Better for
  binary descriptors.

### 5.3 Lowe's Ratio Test — Derivation

The **ratio test** (Lowe, 2004) is the most important heuristic in feature matching.
For each query descriptor $\mathbf{d}_i$, let $d_1$ and $d_2$ be the distances to the
nearest and second-nearest neighbors, respectively.

**Key insight**: For a **correct** match, $d_1$ should be much smaller than $d_2$
(the true correspondence is a clear winner).  For an **incorrect** match, $d_1$ and
$d_2$ are comparable (both are essentially random matches).

#### Probabilistic Model

Lowe modeled the distance distributions as follows:

- **Correct matches**: The distance $d_1$ to the true correspondence follows a
  distribution peaked near zero (approximately Gaussian with small variance $\sigma_c^2$,
  reflecting descriptor noise and geometric distortion):
  $d_1 \sim \mathcal{N}(\mu_c, \sigma_c^2)$ with small $\mu_c$.
  The second-nearest distance $d_2$ is drawn from the background distribution.

- **Incorrect matches**: Both $d_1$ and $d_2$ are drawn from the **background distribution**
  of distances to random (non-matching) descriptors, which is approximately uniform over a
  narrow range in high-dimensional spaces (concentration of measure):
  $d_1, d_2 \sim \text{Uniform}[a, b]$ with $a \approx b$.

The **ratio** $r = d_1 / d_2$ then has:

- For correct matches: $r$ is typically small (peaked near 0), since $d_1 \ll d_2$.
- For incorrect matches: $r$ is close to 1 (both distances are similar order-of-magnitude
  draws from the same background distribution).

#### Distribution of the Ratio

For incorrect matches, both $d_1$ and $d_2$ are order statistics of i.i.d. draws from the
background distribution.  When the background is approximately uniform on $[a, b]$, the
ratio of the first to the second order statistic from $N$ draws concentrates around 1 as
$N$ grows (concentration of measure in high-dimensional descriptor spaces).  Concretely,
the PDF of $r = d_{(1)}/d_{(2)}$ for two uniform draws is $f(r) = 2r$ on $[0, 1]$, which
is heavily weighted toward $r \approx 1$.

For correct matches, $d_1$ is drawn from the "correct" Gaussian while $d_2$ comes from the
background, so $r$ is drawn from a distribution peaked well below 1.  The threshold $\rho$
is chosen where these two distributions separate cleanly.

$$
\text{Accept match if } \frac{d_1}{d_2} < \rho
$$

Lowe (2004) determined the threshold **empirically** by evaluating the ratio distributions
on ground-truth correspondences from known homographies.  At $\rho = 0.75$, approximately
**90% of incorrect matches** are rejected while **95% of correct matches** are retained.
The more conservative $\rho = 0.7$ is also commonly used, trading recall for precision.

### 5.4 Cross-Check Validation

A match $(i, j)$ passes **cross-check** if $j$ is the nearest neighbor of $i$ AND $i$ is
the nearest neighbor of $j$.  This is a simple symmetry test that rejects many false
matches, but is less effective than the ratio test.

### 5.5 Creating a Synthetic Pair for Matching

In [ ]:
np.random.seed(42)
img_a = make_shapes_image(h=300, w=400)

tx, ty = 30, 20
M_warp = np.float32([[1, 0, tx], [0, 1, ty]])
img_b = cv2.warpAffine(img_a, M_warp, (400, 300))
img_b = np.clip(img_b.astype(np.float32) + np.random.randn(300, 400).astype(np.float32) * 10,
                0, 255).astype(np.uint8)

sift = cv2.SIFT_create()
kp_a, des_a = sift.detectAndCompute(img_a, None)
kp_b, des_b = sift.detectAndCompute(img_b, None)
print(f"SIFT keypoints: image A = {len(kp_a)}, image B = {len(kp_b)}")

### 5.6 Brute-Force Matching with Ratio Test

In [ ]:
bf = cv2.BFMatcher(cv2.NORM_L2)
raw_matches = bf.knnMatch(des_a, des_b, k=2)

ratio_threshold = 0.75
good_matches = []
ratios_good, ratios_bad = [], []

for m, n in raw_matches:
    ratio = m.distance / n.distance
    if ratio < ratio_threshold:
        good_matches.append(m)
        ratios_good.append(ratio)
    else:
        ratios_bad.append(ratio)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ratios_good, bins=30, alpha=0.7, label=f"Good ({len(ratios_good)})", color="#009E73")
axes[0].hist(ratios_bad, bins=30, alpha=0.7, label=f"Rejected ({len(ratios_bad)})", color="#D55E00")
axes[0].axvline(ratio_threshold, color="black", linestyle="--", label=f"threshold = {ratio_threshold}")
axes[0].set_xlabel("Ratio $d_1/d_2$")
axes[0].set_ylabel("Count")
axes[0].set_title("Lowe's Ratio Test Distribution")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

good_matches_sorted = sorted(good_matches, key=lambda m: m.distance)
img_matched = cv2.drawMatches(
    img_a, kp_a, img_b, kp_b, good_matches_sorted[:50], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)
axes[1].imshow(img_matched)
axes[1].set_title(f"SIFT matches after ratio test ({len(good_matches)} kept)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

### 5.7 FLANN Matching

In [ ]:
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)

flann = cv2.FlannBasedMatcher(index_params, search_params)
flann_matches = flann.knnMatch(des_a, des_b, k=2)

flann_good = [m for m, n in flann_matches if m.distance < 0.75 * n.distance]

img_flann = cv2.drawMatches(
    img_a, kp_a, img_b, kp_b, sorted(flann_good, key=lambda m: m.distance)[:50],
    None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(14, 5))
plt.imshow(img_flann)
plt.title(f"FLANN matching with ratio test ({len(flann_good)} matches)")
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"BFMatcher good matches: {len(good_matches)}")
print(f"FLANN good matches:     {len(flann_good)}")

---
## 6. RANSAC (Random Sample Consensus)

### 6.1 The Outlier Problem

Feature matching inevitably produces **outliers** — incorrect correspondences.  Even after
the ratio test, 10–30% of matches may be wrong.  Least-squares fitting is catastrophically
sensitive to outliers: a single bad match can corrupt the entire estimate.

**RANSAC** (Fischler & Bolles, 1981) is a robust estimation paradigm that explicitly
handles outliers.

### 6.2 Algorithm

1. **Sample** a minimal set of $s$ correspondences (e.g., $s = 4$ for a homography,
   $s = 8$ for the fundamental matrix).
2. **Fit** the model (e.g., compute homography from 4 points).
3. **Score**: Count inliers — correspondences consistent with the model within a
   distance threshold $\delta$.
4. **Repeat** for $N$ iterations, keeping the model with the most inliers.
5. **Refine**: Re-fit the model using all inliers of the best model.

### 6.3 How Many Iterations?

The full derivation is in the next cell. The key result: for desired success probability $p$,
outlier ratio $\varepsilon$, and minimal sample size $s$:

$$N = \frac{\log(1 - p)}{\log\bigl(1 - (1-\varepsilon)^s\bigr)}$$

This exponential dependence on $s$ motivates using the **minimal solver** (smallest possible $s$).

### 6.4 Beyond RANSAC

Modern robust estimators improve on RANSAC in several ways:
- **MSAC** (Torr & Zisserman, 2000): scores by sum of truncated squared distances instead of inlier count
- **MLESAC**: maximum-likelihood formulation with a mixture of inlier/outlier distributions
- **LO-RANSAC**: applies local optimisation within the RANSAC loop
- **MAGSAC++** (Barath et al., 2020): eliminates the fixed threshold $\delta$ via marginalisation
- **GC-RANSAC** (Barath & Matas, 2018): graph-cut-based local optimisation for spatial consistency

### 6.3 How Many Iterations? — Derivation

Let:
- $\varepsilon$ = outlier ratio (fraction of correspondences that are outliers)
- $s$ = minimal sample size
- $p$ = desired probability of finding at least one all-inlier sample

The probability that a single sample of size $s$ contains **only inliers** is:

$$
q = (1 - \varepsilon)^s
$$

The probability that a single sample has **at least one outlier** is:

$$
1 - q = 1 - (1 - \varepsilon)^s
$$

The probability that **all** $N$ samples have at least one outlier (i.e., RANSAC fails) is:

$$
P(\text{fail}) = \bigl(1 - (1-\varepsilon)^s\bigr)^N
$$

We want $P(\text{fail}) \leq 1 - p$, so:

$$
\bigl(1 - (1-\varepsilon)^s\bigr)^N = 1 - p
$$

Taking logarithms of both sides:

$$
N \cdot \log\bigl(1 - (1-\varepsilon)^s\bigr) = \log(1 - p)
$$

$$
\boxed{N = \frac{\log(1 - p)}{\log\bigl(1 - (1-\varepsilon)^s\bigr)}}
$$

#### Numerical example: $s=8$ (fundamental matrix), $\varepsilon=0.3$, $p=0.99$

$$
N = \frac{\log(1 - 0.99)}{\log(1 - (1-0.3)^8)}
= \frac{\log(0.01)}{\log(1 - 0.7^8)}
= \frac{-4.605}{\log(1 - 0.0576)}
= \frac{-4.605}{-0.0594}
\approx 78
$$

With 50% outliers and $s=8$: $N \approx 1178$ iterations are needed!

### 6.4 Adaptive RANSAC

Instead of fixing $\varepsilon$ beforehand, **adaptive RANSAC** updates the outlier
estimate after each iteration:

1. Initialize $N = \infty$, $\varepsilon = 1$.
2. After finding a model with $n_{\text{in}}$ inliers out of $n_{\text{total}}$:
   $\hat{\varepsilon} = 1 - n_{\text{in}} / n_{\text{total}}$.
3. If $\hat{\varepsilon} < \varepsilon$, update $\varepsilon \leftarrow \hat{\varepsilon}$
   and recompute $N$.
4. Stop when the number of iterations performed exceeds the current $N$.

### 6.5 Iteration Count Visualization

In [ ]:
def compute_ransac_iterations(epsilon, s, p=0.99):
    """Number of RANSAC iterations needed."""
    if epsilon >= 1.0:
        return float("inf")
    q = (1.0 - epsilon) ** s
    if q >= 1.0:
        return 1
    return int(np.ceil(np.log(1.0 - p) / np.log(1.0 - q)))


eps_range = np.linspace(0.05, 0.80, 100)
sample_sizes = [4, 7, 8]

plt.figure(figsize=(10, 5))
for s in sample_sizes:
    N_vals = [compute_ransac_iterations(e, s) for e in eps_range]
    plt.semilogy(eps_range * 100, N_vals, label=f"$s = {s}$", linewidth=2)

plt.xlabel("Outlier ratio $\\varepsilon$ (%)")
plt.ylabel("Required iterations $N$ (log scale)")
plt.title("RANSAC iterations vs. outlier ratio ($p = 0.99$)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.6 RANSAC Implementation (Adaptive)

In [ ]:
def ransac_homography(pts1, pts2, threshold=3.0, max_iter=2000, p=0.99):
    """
    RANSAC for homography estimation with adaptive iteration count.

    Parameters
    ----------
    pts1, pts2 : (N, 2) arrays of corresponding points
    threshold  : inlier distance threshold in pixels
    max_iter   : hard upper bound on iterations
    p          : desired success probability

    Returns
    -------
    best_H      : (3, 3) homography matrix
    inlier_mask : boolean array of length N
    history     : list of inlier counts per iteration
    """
    N = len(pts1)
    best_inliers = 0
    best_mask = np.zeros(N, dtype=bool)
    best_H = None
    history = []

    adaptive_N = max_iter
    rng = np.random.RandomState(42)

    for it in range(max_iter):
        if it >= adaptive_N:
            break

        idx = rng.choice(N, 4, replace=False)
        H, status = cv2.findHomography(pts1[idx].reshape(-1, 1, 2),
                                       pts2[idx].reshape(-1, 1, 2), 0)
        if H is None:
            history.append(best_inliers)
            continue

        pts1_h = np.hstack([pts1, np.ones((N, 1))])  # N x 3
        proj = (H @ pts1_h.T).T  # N x 3
        proj = proj[:, :2] / proj[:, 2:3]

        dists = np.linalg.norm(proj - pts2, axis=1)
        mask = dists < threshold
        n_inliers = mask.sum()

        if n_inliers > best_inliers:
            best_inliers = n_inliers
            best_mask = mask
            best_H = H

            eps = 1.0 - n_inliers / N
            if eps > 0:
                adaptive_N = min(max_iter, compute_ransac_iterations(eps, 4, p))

        history.append(best_inliers)

    if best_inliers >= 4:
        best_H, _ = cv2.findHomography(pts1[best_mask].reshape(-1, 1, 2),
                                       pts2[best_mask].reshape(-1, 1, 2), 0)

    return best_H, best_mask, history

### 6.7 Synthetic Data with Known Inliers/Outliers

In [ ]:
np.random.seed(42)
n_total = 120
n_inliers_true = 80
n_outliers_true = n_total - n_inliers_true

H_true = np.array([[1.05, -0.1, 15],
                    [0.08,  0.97, -10],
                    [0.0001, 0.0002, 1.0]])

pts1_inlier = np.random.rand(n_inliers_true, 2) * np.array([400, 300])
pts1_h = np.hstack([pts1_inlier, np.ones((n_inliers_true, 1))])
proj = (H_true @ pts1_h.T).T
pts2_inlier = proj[:, :2] / proj[:, 2:3] + np.random.randn(n_inliers_true, 2) * 1.0

pts1_outlier = np.random.rand(n_outliers_true, 2) * np.array([400, 300])
pts2_outlier = np.random.rand(n_outliers_true, 2) * np.array([400, 300])

pts1_all = np.vstack([pts1_inlier, pts1_outlier])
pts2_all = np.vstack([pts2_inlier, pts2_outlier])
true_mask = np.array([True] * n_inliers_true + [False] * n_outliers_true)

print(f"Total correspondences: {n_total}")
print(f"True inliers: {n_inliers_true}, True outliers: {n_outliers_true}")
print(f"Outlier ratio: {n_outliers_true/n_total:.1%}")

### 6.8 Running RANSAC

In [ ]:
H_est, inlier_mask, history = ransac_homography(pts1_all, pts2_all, threshold=5.0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(pts1_all[true_mask, 0], pts1_all[true_mask, 1], ".", color="#0072B2", label="True inliers")
axes[0].plot(pts1_all[~true_mask, 0], pts1_all[~true_mask, 1], ".", color="#D55E00", label="True outliers")
axes[0].set_title("Ground truth labels")
axes[0].set_xlabel("x (px)")
axes[0].set_ylabel("y (px)")
axes[0].legend()
axes[0].set_aspect("equal")

axes[1].plot(pts1_all[inlier_mask, 0], pts1_all[inlier_mask, 1], ".", color="#0072B2", label="RANSAC inliers")
axes[1].plot(pts1_all[~inlier_mask, 0], pts1_all[~inlier_mask, 1], ".", color="#D55E00", label="RANSAC outliers")
axes[1].set_title("RANSAC classification")
axes[1].set_xlabel("x (px)")
axes[1].set_ylabel("y (px)")
axes[1].legend()
axes[1].set_aspect("equal")

axes[2].plot(history, linewidth=2, color="#0072B2")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Best inlier count")
axes[2].set_title("RANSAC convergence")
axes[2].axhline(n_inliers_true, color="#009E73", linestyle="--", label="True inlier count")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

precision = (inlier_mask & true_mask).sum() / max(inlier_mask.sum(), 1)
recall = (inlier_mask & true_mask).sum() / true_mask.sum()
print(f"RANSAC found {inlier_mask.sum()} inliers (ground truth: {n_inliers_true})")
print(f"Precision: {precision:.3f}, Recall: {recall:.3f}")
print(f"Converged in {len(history)} iterations")

---
## 7. Fundamental Matrix (8-Point Algorithm)

### 7.1 The Epipolar Constraint

The **fundamental matrix** $F$ encodes the epipolar geometry between two uncalibrated views.
For any pair of corresponding points $\mathbf{x} = (x, y, 1)^\top$ in image 1 and
$\mathbf{x}' = (x', y', 1)^\top$ in image 2 (in homogeneous coordinates):

$$
\mathbf{x}'^\top F \, \mathbf{x} = 0
$$

This says that $\mathbf{x}'$ lies on the **epipolar line** $\ell' = F\mathbf{x}$ in
image 2.  Geometrically, this line is the projection of the ray through $\mathbf{x}$
from camera 1 onto image 2.

#### Degrees of Freedom of $F$

$F$ is a $3 \times 3$ matrix with 9 entries, but its true degrees of freedom are fewer:

1. **Homogeneous scaling** ($9 \to 8$).  The epipolar constraint $\mathbf{x}'^\top F\,\mathbf{x} = 0$
   is unchanged if $F$ is multiplied by any non-zero scalar, so $F$ is defined only
   **up to scale**.  This removes one degree of freedom, leaving **8**.

2. **Rank-2 constraint** ($8 \to 7$).  A valid fundamental matrix satisfies
   $\det(F) = 0$ (see §7.5).  This algebraic constraint removes one more degree of
   freedom, leaving **7 DOF**.

Consequently, **7 point correspondences** are theoretically sufficient to determine $F$
(the *7-point algorithm* solves a cubic and yields 1 or 3 real solutions).  The
**8-point algorithm** (§7.2) uses one extra correspondence to obtain a unique linear
solution at the cost of not enforcing $\det(F) = 0$ automatically — which is why
a separate rank-2 enforcement step is required.

### 7.2 DLT Formulation

Let $F = \begin{bmatrix} f_{11} & f_{12} & f_{13} \\ f_{21} & f_{22} & f_{23} \\ f_{31} & f_{32} & f_{33} \end{bmatrix}$.

Expanding $\mathbf{x}'^\top F \, \mathbf{x} = 0$ for a single correspondence $(\mathbf{x}, \mathbf{x}')$:

$$
x'x \, f_{11} + x'y \, f_{12} + x' \, f_{13} + y'x \, f_{21} + y'y \, f_{22} + y' \, f_{23} + x \, f_{31} + y \, f_{32} + f_{33} = 0
$$

Define the 9-vector:

$$
\mathbf{a} = \begin{bmatrix} x'x & x'y & x' & y'x & y'y & y' & x & y & 1 \end{bmatrix}^\top
$$

and $\mathbf{f} = \text{vec}(F) = \begin{bmatrix} f_{11} & f_{12} & \cdots & f_{33} \end{bmatrix}^\top$.

Then the constraint becomes $\mathbf{a}^\top \mathbf{f} = 0$.

### 7.3 Stacking $N$ Equations

Given $N \geq 8$ correspondences, we stack them into:

$$
A \mathbf{f} = \mathbf{0}, \qquad A \in \mathbb{R}^{N \times 9}
$$

where each row of $A$ is $\mathbf{a}_i^\top$ for correspondence $i$.

### 7.4 SVD Solution

We seek $\mathbf{f}$ that minimizes $\|A\mathbf{f}\|^2$ subject to $\|\mathbf{f}\| = 1$
(to avoid the trivial solution $\mathbf{f} = \mathbf{0}$).

Compute the SVD: $A = U \Sigma V^\top$.  The solution is the **last column of $V$**
(corresponding to the smallest singular value):

$$
\mathbf{f} = \mathbf{v}_9
$$

Reshape $\mathbf{f}$ into $\hat{F}$ (a $3 \times 3$ matrix).

### 7.5 Rank-2 Enforcement

The fundamental matrix must have **rank 2** (i.e., $\det(F) = 0$).  This is because $F$
maps points to epipolar lines, and all epipolar lines pass through the epipole, so the
image of $F$ is a 2-D pencil of lines.

The SVD solution $\hat{F}$ generally has rank 3.  We enforce rank 2 by taking the SVD of
$\hat{F}$ itself:

$$
\hat{F} = U_F \, \text{diag}(\sigma_1, \sigma_2, \sigma_3) \, V_F^\top
$$

and setting the smallest singular value to zero:

$$
F = U_F \, \text{diag}(\sigma_1, \sigma_2, 0) \, V_F^\top
$$

This is the closest rank-2 matrix to $\hat{F}$ in the Frobenius norm (Eckart–Young–Mirsky
theorem).

### 7.6 Hartley Normalization

The 8-point algorithm is extremely sensitive to the coordinate system of the input points.
**Hartley normalization** (1997) transforms both point sets to have:

- **Zero mean**: $\bar{\mathbf{x}} = \mathbf{0}$
- **Average distance from origin** $= \sqrt{2}$

The normalization matrix for a set of points $\{(x_i, y_i)\}$ with centroid $(\bar{x}, \bar{y})$
and mean distance $\bar{d}$ is:

$$
T = \begin{bmatrix} s & 0 & -s\bar{x} \\ 0 & s & -s\bar{y} \\ 0 & 0 & 1 \end{bmatrix},
\qquad s = \frac{\sqrt{2}}{\bar{d}}
$$

After computing $\hat{F}$ from the normalized points $\tilde{\mathbf{x}} = T\mathbf{x}$
and $\tilde{\mathbf{x}}' = T'\mathbf{x}'$, the fundamental matrix in original coordinates is:

$$
F = T'^\top \hat{F} \, T
$$

This simple pre-conditioning step dramatically improves numerical stability.

### 7.7 Implementation

In [ ]:
def normalize_points(pts):
    """
    Hartley normalization: zero mean, average distance = sqrt(2).

    Returns: normalized points (N, 2), normalization matrix T (3, 3)
    """
    mean = pts.mean(axis=0)
    centered = pts - mean
    mean_dist = np.mean(np.linalg.norm(centered, axis=1))
    s = np.sqrt(2.0) / (mean_dist + 1e-12)

    T = np.array([[s, 0, -s * mean[0]],
                  [0, s, -s * mean[1]],
                  [0, 0, 1]], dtype=np.float64)

    pts_h = np.hstack([pts, np.ones((len(pts), 1))])
    pts_norm = (T @ pts_h.T).T
    return pts_norm[:, :2], T


def eight_point_algorithm(pts1, pts2):
    """
    Normalized 8-point algorithm for fundamental matrix estimation.

    Parameters
    ----------
    pts1, pts2 : (N, 2) arrays of corresponding points, N >= 8

    Returns
    -------
    F : (3, 3) fundamental matrix
    """
    pts1_n, T1 = normalize_points(pts1)
    pts2_n, T2 = normalize_points(pts2)

    N = len(pts1)
    x1, y1 = pts1_n[:, 0], pts1_n[:, 1]
    x2, y2 = pts2_n[:, 0], pts2_n[:, 1]

    A = np.column_stack([
        x2 * x1, x2 * y1, x2,
        y2 * x1, y2 * y1, y2,
        x1, y1, np.ones(N)
    ])

    _, _, Vt = np.linalg.svd(A)
    F_hat = Vt[-1].reshape(3, 3)

    U_f, S_f, Vt_f = np.linalg.svd(F_hat)
    S_f[2] = 0.0
    F_hat = U_f @ np.diag(S_f) @ Vt_f

    F = T2.T @ F_hat @ T1
    F /= F[2, 2] + 1e-15

    return F

### 7.8 Testing on Synthetic Two-View Data

In [ ]:
pts3d, K, R_true, t_true, pts1_2v, pts2_2v = make_two_view_scene(n_points=80, noise_px=0.5)

F_ours = eight_point_algorithm(pts1_2v, pts2_2v)

epipolar_errors = []
for i in range(len(pts1_2v)):
    x1_h = np.array([pts1_2v[i, 0], pts1_2v[i, 1], 1.0])
    x2_h = np.array([pts2_2v[i, 0], pts2_2v[i, 1], 1.0])
    err = np.abs(x2_h @ F_ours @ x1_h)
    epipolar_errors.append(err)

print(f"Mean epipolar error: {np.mean(epipolar_errors):.6f}")
print(f"Max  epipolar error: {np.max(epipolar_errors):.6f}")
print(f"\nEstimated F (our 8-point):")
print(F_ours)
print(f"\nRank of F: {np.linalg.matrix_rank(F_ours, tol=1e-6)}")

### 7.10 Drawing Epipolar Lines

In [ ]:
def draw_epipolar_lines(img_shape, F, pts1, pts2, n_lines=15):
    """
    Visualize epipolar lines in both images.
    """
    h, w = img_shape
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    colors = plt.cm.tab20(np.linspace(0, 1, n_lines))
    rng = np.random.RandomState(0)
    indices = rng.choice(len(pts1), min(n_lines, len(pts1)), replace=False)

    for ax_idx, (pts_draw, pts_other, F_use) in enumerate([
        (pts2, pts1, F),       # epipolar lines in image 2
        (pts1, pts2, F.T),     # epipolar lines in image 1
    ]):
        ax = axes[ax_idx]
        ax.set_xlim(0, w)
        ax.set_ylim(h, 0)
        ax.set_aspect("equal")
        ax.set_xlabel("x (px)")
        ax.set_ylabel("y (px)")
        ax.set_title(f"Image {ax_idx + 1}: epipolar lines")

        for k, i in enumerate(indices):
            x_other = np.array([pts_other[i, 0], pts_other[i, 1], 1.0])
            line = F_use @ x_other  # ax + by + c = 0
            a, b, c = line

            if abs(b) > 1e-10:
                x_vals = np.array([0, w])
                y_vals = -(a * x_vals + c) / b
            else:
                y_vals = np.array([0, h])
                x_vals = -(b * y_vals + c) / a

            ax.plot(x_vals, y_vals, color=colors[k], linewidth=1, alpha=0.7)
            ax.plot(pts_draw[i, 0], pts_draw[i, 1], "o", color=colors[k], markersize=5)

    plt.suptitle("Epipolar Lines from Estimated $F$", fontsize=14)
    plt.tight_layout()
    plt.show()


draw_epipolar_lines((480, 640), F_ours, pts1_2v, pts2_2v, n_lines=15)

### 7.11 Verifying the Epipolar Constraint Numerically

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(epipolar_errors, bins=25, edgecolor="black", alpha=0.7)
plt.xlabel("$|\\mathbf{x}'^T F \\mathbf{x}|$")
plt.ylabel("Count")
plt.title("Epipolar constraint errors ($\\approx 0$ means good)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Essential Matrix

### 8.1 Relation to the Fundamental Matrix

The essential matrix was introduced by **Longuet-Higgins (1981)** — one of the foundational
papers in computer vision, published a decade before the fundamental matrix was formalized
by Faugeras (1992) and Hartley (1992). The key insight: when camera intrinsics $K_1, K_2$
are known, the epipolar constraint simplifies from a projective relationship to a
*Euclidean* one, directly encoding the relative rotation and translation.

$$
E = K_2^\top F K_1
$$

The essential matrix relates **normalized image coordinates** (calibrated coordinates)
$\hat{\mathbf{x}} = K^{-1} \mathbf{x}$:

$$
\hat{\mathbf{x}}'^\top E \, \hat{\mathbf{x}} = 0
$$

In practice, most modern systems use the essential matrix (not F), because camera
calibration is typically available. The **5-point algorithm** (Nistér, 2004) is the
standard minimal solver — it exploits the additional constraints of $E$ to require
only 5 correspondences (vs. 8 for F), reducing RANSAC iterations significantly.

### 8.2 Properties of the Essential Matrix

The essential matrix $E$ satisfies:

1. **Rank 2**: $\det(E) = 0$
2. **Two equal non-zero singular values**: $E = U \, \text{diag}(\sigma, \sigma, 0) \, V^\top$

**Proof.** Since $E = [\mathbf{t}]_\times R$: (1) $\text{rank}([\mathbf{t}]_\times) = 2$
(its null space is $\mathbf{t}$) and $R$ is full rank, so $\text{rank}(E) = 2$.
(2) For the singular values, compute $E^\top E = R^\top [\mathbf{t}]_\times^\top [\mathbf{t}]_\times R$.
Now $[\mathbf{t}]_\times^\top [\mathbf{t}]_\times = \|\mathbf{t}\|^2 I - \mathbf{t}\mathbf{t}^\top$
(verify by expanding the skew-symmetric product). Its eigenvalues are $\|\mathbf{t}\|^2$
(multiplicity 2) and $0$ (eigenvector $\mathbf{t}$). Since $R$ is orthogonal,
$E^\top E$ has the same eigenvalues $(\|\mathbf{t}\|^2, \|\mathbf{t}\|^2, 0)$,
giving $\sigma_1 = \sigma_2 = \|\mathbf{t}\|$, $\sigma_3 = 0$. $\square$

### 8.3 Decomposition into $(R, \mathbf{t})$

Given the SVD $E = U \, \text{diag}(\sigma, \sigma, 0) \, V^\top$, there are exactly
**four** possible decompositions into rotation and translation:

Define the auxiliary matrix:

$$
W = \begin{bmatrix} 0 & -1 & 0 \\ 1 & 0 & 0 \\ 0 & 0 & 1 \end{bmatrix}
$$

Note that $W$ is a 90° rotation about the $z$-axis.  The four solutions are:

$$
R_1 = U W V^\top, \quad R_2 = U W^\top V^\top, \quad \mathbf{t} = \pm \mathbf{u}_3
$$

where $\mathbf{u}_3$ is the third column of $U$.

**Why four solutions?** The essential matrix is defined up to sign ($E$ and $-E$ yield
the same epipolar geometry), and the factorization $E = [\mathbf{t}]_\times R$ has a
two-fold ambiguity (called the **twisted pair** ambiguity): both $(R_1, \mathbf{t})$
and $(R_2, -\mathbf{t})$ produce the same essential matrix.  Combined with the
$\pm\mathbf{t}$ ambiguity from the sign, we get $2 \times 2 = 4$ solutions.

Note: We enforce $\det(R) = +1$ by negating $R$ if $\det(R) = -1$.

### 8.4 Cheirality Check

Only **one** of the four solutions places the triangulated 3-D points **in front of both
cameras** (positive depth).  This is the **cheirality constraint**.

To check: triangulate a point using the candidate $(R, \mathbf{t})$, then verify that the
$z$-coordinate is positive in both camera frames:

- In camera 1: $z_1 > 0$
- In camera 2: the point transformed by $(R, \mathbf{t})$ has $z_2 > 0$

We test all four solutions and select the one where the most points pass the cheirality
check.

### 8.5 Implementation

In [ ]:
def decompose_essential(E):
    """
    Decompose E into four (R, t) candidates.
    """
    U, S, Vt = np.linalg.svd(E)

    if np.linalg.det(U) < 0:
        U = -U
    if np.linalg.det(Vt) < 0:
        Vt = -Vt

    W = np.array([[0, -1, 0],
                  [1,  0, 0],
                  [0,  0, 1]], dtype=np.float64)

    R1 = U @ W @ Vt
    R2 = U @ W.T @ Vt
    t = U[:, 2].reshape(3, 1)

    solutions = [
        (R1,  t),
        (R1, -t),
        (R2,  t),
        (R2, -t),
    ]
    return solutions


def triangulate_point(P1, P2, x1, x2):
    """
    Linear triangulation (DLT) for a single point correspondence.
    """
    A = np.array([
        x1[0] * P1[2] - P1[0],
        x1[1] * P1[2] - P1[1],
        x2[0] * P2[2] - P2[0],
        x2[1] * P2[2] - P2[1],
    ])
    _, _, Vt = np.linalg.svd(A)
    X = Vt[-1]
    return X[:3] / X[3]


def cheirality_check(K, R, t, pts1, pts2):
    """
    Count how many points are in front of both cameras.
    """
    P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
    P2 = K @ np.hstack([R, t])

    n_front = 0
    for i in range(len(pts1)):
        X = triangulate_point(P1, P2, pts1[i], pts2[i])

        z1 = X[2]
        X_cam2 = R @ X + t.ravel()
        z2 = X_cam2[2]

        if z1 > 0 and z2 > 0:
            n_front += 1

    return n_front

### 8.6 Computing E and Applying the Cheirality Check

In [ ]:
E_ours = K.T @ F_ours @ K

U_e, S_e, Vt_e = np.linalg.svd(E_ours)
print("Singular values of E (should be [σ, σ, 0]):")
print(S_e)

sigma_avg = (S_e[0] + S_e[1]) / 2.0
E_corrected = U_e @ np.diag([sigma_avg, sigma_avg, 0]) @ Vt_e

solutions = decompose_essential(E_corrected)

print("\nCheirality check for 4 solutions:")
best_n = 0
best_idx = 0
for idx, (R_cand, t_cand) in enumerate(solutions):
    n = cheirality_check(K, R_cand, t_cand, pts1_2v, pts2_2v)
    det_sign = "+" if np.linalg.det(R_cand) > 0 else "-"
    print(f"  Solution {idx}: {n:3d}/{len(pts1_2v)} in front, det(R) = {det_sign}1")
    if n > best_n:
        best_n = n
        best_idx = idx

R_est, t_est = solutions[best_idx]
print(f"\nBest solution: #{best_idx} ({best_n} points in front of both cameras)")

### 8.7 Comparing Recovered Pose to Ground Truth

In [ ]:
t_true_normalized = t_true.ravel() / np.linalg.norm(t_true)
t_est_normalized = t_est.ravel() / np.linalg.norm(t_est)

if np.dot(t_true_normalized, t_est_normalized) < 0:
    t_est_normalized = -t_est_normalized

print("Recovered R:")
print(R_est)
print("\nTrue R:")
print(R_true)
print(f"\nRotation error (Frobenius): {np.linalg.norm(R_est - R_true):.6f}")

print(f"\nRecovered t (normalized): {t_est_normalized}")
print(f"True t (normalized):      {t_true_normalized}")

cos_angle = np.clip(np.abs(np.dot(t_est_normalized, t_true_normalized)), -1, 1)
print(f"Translation direction error: {np.arccos(cos_angle) * 180 / np.pi:.4f}°")

### 8.9 Triangulation and 3-D Reconstruction

In [ ]:
P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
P2 = K @ np.hstack([R_est, t_est])

pts3d_reconstructed = []
for i in range(len(pts1_2v)):
    X = triangulate_point(P1, P2, pts1_2v[i], pts2_2v[i])
    pts3d_reconstructed.append(X)
pts3d_reconstructed = np.array(pts3d_reconstructed)

scale = np.linalg.norm(t_true) / np.linalg.norm(t_est)
pts3d_scaled = pts3d_reconstructed * scale

fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(121, projection="3d")
ax1.scatter(pts3d[:, 0], pts3d[:, 1], pts3d[:, 2], c="blue", s=10, alpha=0.6)
ax1.set_title("Ground-truth 3D points")
ax1.set_xlabel("X")
ax1.set_ylabel("Y")
ax1.set_zlabel("Z")
ax1.view_init(elev=25, azim=135)

ax2 = fig.add_subplot(122, projection="3d")
ax2.scatter(pts3d_scaled[:, 0], pts3d_scaled[:, 1], pts3d_scaled[:, 2],
            c="red", s=10, alpha=0.6)
ax2.set_title("Reconstructed 3D points (scaled)")
ax2.set_xlabel("X")
ax2.set_ylabel("Y")
ax2.set_zlabel("Z")
ax2.view_init(elev=25, azim=135)

plt.suptitle("Triangulation: Ground Truth vs. Reconstruction", fontsize=13)
plt.tight_layout()
plt.show()

### 8.10 Reprojection Error Analysis

In [ ]:
reproj_errors_1, reproj_errors_2 = [], []
for i in range(len(pts1_2v)):
    X_h = np.append(pts3d_reconstructed[i], 1.0)
    p1_proj = P1 @ X_h
    p1_proj = p1_proj[:2] / p1_proj[2]
    reproj_errors_1.append(np.linalg.norm(p1_proj - pts1_2v[i]))

    p2_proj = P2 @ X_h
    p2_proj = p2_proj[:2] / p2_proj[2]
    reproj_errors_2.append(np.linalg.norm(p2_proj - pts2_2v[i]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(reproj_errors_1, bins=25, edgecolor="black", alpha=0.7, color="#0072B2")
axes[0].set_xlabel("Reprojection error (px)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Image 1 (mean = {np.mean(reproj_errors_1):.3f} px)")
axes[0].grid(True, alpha=0.3)
axes[1].hist(reproj_errors_2, bins=25, edgecolor="black", alpha=0.7, color="#D55E00")
axes[1].set_xlabel("Reprojection error (px)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Image 2 (mean = {np.mean(reproj_errors_2):.3f} px)")
axes[1].grid(True, alpha=0.3)
plt.suptitle("Triangulation Reprojection Errors", fontsize=13)
plt.tight_layout()
plt.show()

---
## 9. Learned Features (Discussion + API Demo)

### 9.1 The Paradigm Shift

Classical features (Harris, SIFT, ORB) rely on **hand-crafted** rules for detection and
description.  These were designed by humans with expert knowledge of image processing.
Since ~2018, **learned features** have consistently outperformed classical methods on
standard benchmarks, especially in challenging conditions:

- Poor texture (indoor scenes, white walls)
- Repetitive patterns (building facades, tiled floors)
- Large viewpoint / illumination changes
- Motion blur

### 9.2 SuperPoint (DeTone et al., 2018)

**SuperPoint** is a **self-supervised** CNN that jointly detects keypoints and computes
descriptors in a single forward pass.

**Architecture**: Shared encoder (VGG-like) → two decoder heads:
- **Keypoint head**: outputs a dense heatmap, trained via *Homographic Adaptation*
  (warp the image by random homographies, detect corners in each warped version,
  aggregate to get pseudo-ground-truth)
- **Descriptor head**: outputs a dense descriptor map (interpolate at keypoint locations)

**Key insight**: Homographic adaptation provides **self-labeling** — no manual keypoint
annotations needed.

### 9.3 LightGlue (Lindenberger et al., 2023)

**LightGlue** is a lightweight learned feature matcher, the successor to **SuperGlue**.
It uses a **transformer** architecture with self- and cross-attention layers to reason
about keypoint matches.

Key improvements over SuperGlue:
- **Adaptive early stopping**: exits when confident, reducing compute for easy pairs
- **Lighter architecture**: fewer parameters, faster inference
- **Flash attention**: efficient attention implementation

Typical pipeline: **SuperPoint + LightGlue** for sparse matching.

### 9.4 LoFTR — Architecture Deep Dive (Sun et al., 2021)

**LoFTR** (Local Feature TRansformer) takes a fundamentally different approach:
**detector-free, semi-dense matching**.  Instead of detect → describe → match,
LoFTR processes both images jointly from the start.

**Architecture pipeline:**

1. **CNN backbone** (ResNet-like FPN): each image is encoded into a feature map at
   **1/8 resolution** — an 800×600 image becomes a 100×75 grid of 256-dim feature
   vectors.
2. **Flatten & positional encoding**: the 2-D feature maps are flattened into 1-D
   sequences (like NLP tokens) with sinusoidal positional encodings added so the
   transformer knows *where* each token lives spatially.
3. **Self-attention + cross-attention** (repeated $L$ times):
   - *Self-attention* within each image refines features by attending to the global
     context of the same image.
   - *Cross-attention* between the two images lets each feature vector in image A
     attend to *every* feature vector in image B, and vice-versa.
4. **Coarse matching**: compute a score matrix
   $\mathbf{S}(i,j) = \text{softmax}(\tilde{\mathbf{F}}_A \tilde{\mathbf{F}}_B^\top)$,
   then extract **mutual nearest neighbors** — a pair $(i,j)$ is kept only if $i$ is
   $j$'s best match *and* $j$ is $i$'s best match.
5. **Fine refinement**: for each coarse match, crop a small window in the fine (1/2)
   resolution feature maps and compute a local correlation volume to refine the match
   location to **sub-pixel accuracy**.

**The key insight**: cross-attention between the two images lets each pixel "look at"
all pixels in the other image.  This gives LoFTR a global receptive field across images,
enabling reliable matching in **textureless regions** where no keypoint detector would
fire — the network infers correspondences from surrounding context rather than local
appearance alone.

**Attention mechanism** — each transformer layer computes:

$$
\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V})
= \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d}}\right)\mathbf{V}
$$

where $\mathbf{Q}, \mathbf{K}, \mathbf{V}$ are linear projections of the input features
and $d$ is the head dimension.  For *self-attention*, $\mathbf{Q}, \mathbf{K}, \mathbf{V}$
all come from the same image; for *cross-attention*, $\mathbf{Q}$ comes from image A while
$\mathbf{K}, \mathbf{V}$ come from image B (and vice-versa in the next sub-layer).

**Efficient LoFTR** (Wang et al., CVPR 2024) addresses the quadratic cost of full
attention on high-resolution feature maps:
- **Aggregated attention**: replaces full $O(N^2)$ attention with an efficient two-step
  aggregate-then-broadcast scheme that reduces complexity to $O(N)$.
- **Adaptive token selection**: dynamically prunes uninformative tokens (e.g., sky
  regions) to further reduce computation.
- Result: **~2.5× speedup** over the original LoFTR with comparable or better accuracy
  on MegaDepth and ScanNet benchmarks, making semi-dense matching practical for
  near-real-time applications.

### 9.5 RoMa / RoMa v2 — Dense Matching (Edstedt et al., 2024–2026)

**RoMa** (Robust Dense Matching) establishes a correspondence for **every pixel**
in image A to a location in image B, producing a dense warp field plus a per-pixel
confidence map.

**Architecture:**

1. **Frozen DINOv2 backbone**: both images are passed through a frozen DINOv2 ViT to
   extract coarse features.  Using a *foundation model* frozen at inference time gives
   strong robustness to appearance changes (day/night, season, weather) because DINOv2
   was trained on hundreds of millions of diverse images.
2. **Coarse matching via feature correlation**: DINOv2 features from image A are matched
   to image B features using a 4-D correlation volume, producing coarse (low-resolution)
   warp estimates.
3. **Learned convolutional decoder**: a multi-scale convolutional decoder progressively
   upsamples the coarse warp field to **full pixel resolution**, refining the match
   locations at each level.  This decoder is trained end-to-end on ground-truth
   correspondences from datasets like MegaDepth.
4. **Confidence head**: a parallel branch predicts per-pixel match confidence, enabling
   downstream selection of the most reliable matches.

**Dense output**: unlike sparse or semi-dense methods that return *selected*
correspondences, RoMa returns a match for **every single pixel**.  The output is a warp
tensor of shape $(H, W, 4)$ mapping each $(x_A, y_A)$ to $(x_B, y_B)$, plus a
$(H, W)$ confidence map.  This is ideal for dense MVS reconstruction.

**RoMa v2 (2026)** upgrades the backbone to **DINOv3**, the latest generation of
self-supervised vision transformers, achieving new SOTA results:
- **MegaDepth-1500**: AUC@5° ≈ 74.2 (+2.1 over v1)
- **ScanNet-1500**: AUC@5° ≈ 68.8 (+1.6 over v1)
- Improved scale-equivariance handling via multi-resolution DINOv3 features

### 9.6 Matching Paradigm Taxonomy

Modern feature matching falls into three paradigms that trade off **match density**
against **computational cost**:

---

**(1) Sparse matching** — detect keypoints → compute descriptors → match

Examples: SIFT, ORB, SuperPoint + LightGlue

- Detect a small set of salient keypoints (corners, blobs) in each image.
- Compute a local descriptor vector at each keypoint.
- Match descriptors across images (nearest-neighbor + ratio test / learned matcher).
- Typically **500–2000 matches** per pair.
- **Fastest** paradigm (~15–25 ms on GPU for SuperPoint + LightGlue).
- Best for: real-time visual odometry, SLAM, fast SfM.

**(2) Semi-dense matching** — match on a downsampled feature grid, refine

Examples: LoFTR, Efficient LoFTR, ASpanFormer

- Skip the detection step entirely.  Instead, extract features on a regular grid
  (e.g., every 8th pixel) and let the network decide which grid cells correspond.
- Transformer cross-attention enables matches even where no corner or blob exists.
- Typically **2,000–5,000 matches** per pair, spread across textureless regions too.
- **Medium speed** (~40–80 ms on GPU).
- Best for: wide-baseline pairs, textureless indoor scenes, outdoor SfM.

**(3) Dense matching** — predict a correspondence for *every* pixel

Examples: DKM, RoMa, RoMa v2

- Output a full warp field mapping every pixel in image A to image B, plus a
  per-pixel confidence map.
- Downstream tasks sample the top-$k$ most confident matches.
- Typically **10k+ matches** available; the entire warp is usable for MVS.
- **Slowest** paradigm (~150–300 ms on GPU).
- Best for: multi-view stereo, dense 3-D reconstruction, wide-baseline pairs with
  significant appearance change.

---

**The fundamental tradeoff — density vs. speed:**

| Paradigm | Matches | GPU Time | Use Case |
|:---|:---:|:---:|:---|
| Sparse (SP + LG) | ~1k | ~20 ms | Real-time VO / SLAM |
| Semi-dense (LoFTR) | ~3k | ~60 ms | Offline SfM, textureless |
| Dense (RoMa v2) | ~10k+ | ~200 ms | Dense MVS, extreme viewpoint |

**Rule of thumb**: start with sparse matching.  If matches are too few (textureless
scenes) or the baseline is very wide, escalate to semi-dense.  Use dense matching when
you need per-pixel correspondences or the viewpoint/appearance gap is extreme.

### 9.8 LoMa (2026) — Pushing Sparse Matching to New Limits

**LoMa** (Local Matching) demonstrates that the classical sparse paradigm —
detect → describe → match — can still compete with dense methods when each
component uses modern designs and a rigorous training recipe.

**Components:**
- **DeDoDe** keypoint detector + descriptor: a learned detector/descriptor pair that
  produces repeatable, distinctive keypoints with 256-dim descriptors.
- **LightGlue** matcher: the efficient transformer matcher from §9.3, applied on
  DeDoDe features instead of SuperPoint.
- **Modern training recipe**: large-scale training on MegaDepth with hard-negative
  mining, multi-resolution augmentation, and longer schedules.

**Results (2026 benchmarks):**
- **SOTA on HardMatch**: a new benchmark specifically targeting difficult matching
  scenarios (repetitive texture, extreme illumination change).
- **SOTA on WxBS** (Wide-baseline Stereo): challenging outdoor pairs with very
  large viewpoint differences.
- **SOTA on InLoc**: indoor localization benchmark requiring precise 6-DoF pose
  under significant appearance change.

**Takeaway**: LoMa shows the sparse paradigm is far from saturated — modern
detectors, descriptors, and matchers trained at scale can rival or beat semi-dense
and dense methods on hard benchmarks, while retaining the speed advantage of sparse
matching.

### 9.9 Connection to DUSt3R & MASt3R — Implicit Dense Matching

The methods above (LoFTR, RoMa, etc.) produce 2-D correspondences that are then fed to
classical geometric estimation (essential matrix, PnP, triangulation).  A newer family of
models **bypasses explicit matching entirely** by predicting 3-D structure directly.

**DUSt3R** (Wang et al., CVPR 2024) takes two images and outputs **dense 3-D pointmaps**
— for every pixel in each image, the network predicts its 3-D position in a shared
coordinate frame.  Under the hood, DUSt3R is *implicitly* performing dense matching:
its cross-attention layers learn which pixels across the two views correspond, and the
network encodes this into consistent 3-D predictions.

**MASt3R** (Leroy et al., 2024) extends DUSt3R by adding an explicit **local feature
head** on top of the same architecture.  This produces per-pixel descriptors that can be
used for fast matching and retrieval, bridging the gap between direct 3-D regression and
classical feature-matching pipelines.

**Why this matters for matching:**
- DUSt3R/MASt3R **subsume** the matching step: you get 3-D geometry without ever
  computing an explicit set of 2-D correspondences.
- They handle **uncalibrated** cameras — no intrinsics needed at inference time.
- The dense 3-D pointmaps can be used to *derive* correspondences if needed (e.g., for
  bundle adjustment initialization).
- These models represent the frontier where **matching and reconstruction merge** into a
  single learned pipeline, a theme explored further in our DUSt3R/MASt3R notebook
  (Notebook 11).

> **Preview**: in Notebook 11 we'll use DUSt3R and MASt3R to reconstruct scenes
> end-to-end from unposed images — effectively replacing the entire detect → match →
> estimate geometry → triangulate pipeline with a single forward pass.

### 9.10 API Usage Patterns

Below we show the typical interfaces for each library.  These require model weights
and PyTorch to run but illustrate the calling conventions.

In [ ]:
superpoint_lightglue_example = """
# --- SuperPoint + LightGlue (via kornia or hloc) ---
from kornia.feature import SuperPoint, LightGlue
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize
sp = SuperPoint(max_num_keypoints=2048).to(device)
lg = LightGlue(features="superpoint").to(device)

# Detect & describe
img0_tensor = kornia.image_to_tensor(img0).float() / 255.0
img1_tensor = kornia.image_to_tensor(img1).float() / 255.0

feats0 = sp.extract(img0_tensor.to(device))
feats1 = sp.extract(img1_tensor.to(device))

# Match
matches = lg({"image0": feats0, "image1": feats1})
mkpts0 = feats0["keypoints"][matches["matches"][:, 0]]
mkpts1 = feats1["keypoints"][matches["matches"][:, 1]]
confidence = matches["scores"]
"""

loftr_example = """
# --- LoFTR (detector-free, semi-dense matching) ---
from kornia.feature import LoFTR

loftr = LoFTR(pretrained="outdoor").to(device)

input_dict = {
    "image0": img0_tensor.to(device),  # (1, 1, H, W) grayscale
    "image1": img1_tensor.to(device),
}
with torch.no_grad():
    result = loftr(input_dict)

mkpts0 = result["keypoints0"].cpu().numpy()  # (N, 2)
mkpts1 = result["keypoints1"].cpu().numpy()
confidence = result["confidence"].cpu().numpy()
"""

roma_example = """
# --- RoMa v2 (dense matching) ---
from romatch import roma_outdoor

model = roma_outdoor(device=device)

warp, certainty = model.match(img_path_A, img_path_B)
# warp: (H, W, 4) -- each pixel maps to (x1, y1, x2, y2)
# certainty: (H, W) confidence map

# Sample top-k matches by confidence
matches, confidence = model.sample(warp, certainty, num=5000)
mkpts0 = matches[:, :2]  # (N, 2) in image 0
mkpts1 = matches[:, 2:]  # (N, 2) in image 1
"""

print("SuperPoint + LightGlue API:")
print(superpoint_lightglue_example)
print("\nLoFTR API:")
print(loftr_example)
print("\nRoMa v2 API:")
print(roma_example)

### 9.11 Performance Landscape

The following summarizes approximate performance on the HPatches benchmark (homography
estimation AUC) and MegaDepth (relative pose estimation AUC):

| Method | Type | HPatches AUC@3px | MegaDepth AUC@5° | Speed | Year |
|:---|:---|:---:|:---:|:---:|:---:|
| SIFT + mutual NN | Sparse | 52.3 | 42.1 | Fast | 2004 |
| ORB + BFMatcher | Sparse | 38.1 | 28.4 | **Fastest** | 2011 |
| SuperPoint + SuperGlue | Sparse | 68.2 | 63.5 | Medium | 2020 |
| SuperPoint + LightGlue | Sparse | 70.1 | 65.8 | Medium | 2023 |
| **XFeat** | Sparse/Semi | 66.8 | 59.1 | **5× faster** | 2024 |
| **XFeat + LighterGlue** | Sparse | 69.5 | 63.3 | Fast | 2024 |
| LoFTR | Semi-dense | 72.4 | 67.3 | Slow | 2021 |
| Efficient LoFTR | Semi-dense | 72.1 | 67.5 | Medium | 2024 |
| OmniGlue | Sparse | 67.5 | 64.2 | Medium | 2024 |
| LoMa (DeDoDe + LG) | Sparse | 73.6 | 69.4 | Medium | 2026 |
| RoMa v1 | Dense | 76.8 | 72.1 | Slow | 2024 |
| RoMa v2 | Dense | **78.3** | **74.2** | Slow | 2026 |

*Numbers are approximate and depend on evaluation protocol.*

**Key takeaways**:
- Learned methods consistently outperform classical ones, especially for large viewpoint changes
- Dense methods (RoMa) achieve the best raw accuracy but are the slowest
- LoMa (2026) shows that modern sparse pipelines can approach semi-dense accuracy at sparse speeds
- Efficient LoFTR matches original LoFTR accuracy at ~2.5× the speed
- SuperPoint + LightGlue offers the best speed/accuracy trade-off for real-time applications
- OmniGlue trades peak accuracy for cross-domain generalization
- Classical ORB remains viable for resource-constrained scenarios (embedded, mobile)
- **XFeat** (CVPR 2024) targets embedded/CPU deployment — 5× faster than SuperPoint with competitive accuracy; pairs with **LighterGlue** (a pruned LightGlue, 3× faster than original)

**Survey references**:
- "Deep Learning Reforms Image Matching: A Survey and Outlook" (arXiv 2506.04619, 2025) — the most comprehensive taxonomy covering sparse, semi-dense, dense matchers and pose regressors
- "Cross-View Feature Matching: Survey with Foundation-Model Perspectives" (arXiv 2608.11093, 2026) — adds DINOv2/v3 and VFM-based matching to the picture

---
## 10. Exercises

The following exercises reinforce the core concepts from this notebook.  Skeleton code is
provided with `TODO` markers where you need to fill in the implementation.

---

### Exercise 1: Implement Harris Corner Detector from Scratch

Complete the function below **without using `cv2.cornerHarris`**. You may use
`cv2.Sobel` for gradients and `scipy.ndimage.gaussian_filter` for smoothing.

In [ ]:
def harris_exercise(img, k=0.04, sigma=1.5, threshold_ratio=0.01):
    """
    Exercise 1: Implement the Harris corner detector.

    Steps:
    1. Compute image gradients Ix, Iy using Sobel operators
    2. Compute products Ix^2, Iy^2, Ix*Iy
    3. Apply Gaussian smoothing to get structure tensor components
    4. Compute Harris response R = det(M) - k * trace(M)^2
    5. Threshold and apply non-maximum suppression

    Parameters
    ----------
    img : 2-D uint8 array
    k : Harris free parameter (default 0.04)
    sigma : Gaussian smoothing sigma
    threshold_ratio : keep R > threshold_ratio * max(R)

    Returns
    -------
    corners : (N, 2) array of (row, col)
    R : response map
    """
    from scipy.ndimage import gaussian_filter, maximum_filter
    img_f = img.astype(np.float64)

    # Step 1 — Gradients
    Ix = cv2.Sobel(img_f, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(img_f, cv2.CV_64F, 0, 1, ksize=3)

    # Step 2 — Structure tensor M = G_σ * [Ix², IxIy; IxIy, Iy²]
    Ixx = gaussian_filter(Ix * Ix, sigma=sigma)
    Ixy = gaussian_filter(Ix * Iy, sigma=sigma)
    Iyy = gaussian_filter(Iy * Iy, sigma=sigma)

    # Step 3 — Harris response: R = det(M) - k·trace(M)²
    det_M = Ixx * Iyy - Ixy * Ixy
    trace_M = Ixx + Iyy
    R = det_M - k * (trace_M ** 2)

    # Step 4 — Threshold + NMS
    R_max = R.max()
    R_thresh = R.copy()
    R_thresh[R < threshold_ratio * R_max] = 0

    # Keep pixels that are local maxima in a 3×3 neighbourhood
    local_max = maximum_filter(R_thresh, size=3)
    corner_mask = (R_thresh > 0) & (R_thresh == local_max)
    rows, cols = np.where(corner_mask)
    corners = np.column_stack([rows, cols])

    assert R_max > 0, "Harris response should be non-zero on textured input"
    if len(corners) > 0:
        assert np.all(R[corners[:, 0], corners[:, 1]] > 0), \
            "All corners must exceed the response threshold"

    return corners, R

In [ ]:
test_img_ex1 = make_checkerboard(rows=6, cols=6, square_size=40)
corners_ex1, R_ex1 = harris_exercise(test_img_ex1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(R_ex1, cmap="jet")
axes[0].set_title("Your Harris response")
axes[1].imshow(test_img_ex1, cmap="gray")
if len(corners_ex1) > 0:
    axes[1].plot(corners_ex1[:, 1], corners_ex1[:, 0], "r+", markersize=8)
axes[1].set_title(f"Your detected corners ({len(corners_ex1)})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---

### Exercise 2: ORB Feature Matching with Ratio Test

Detect ORB features in two synthetic views, match them using `cv2.BFMatcher` with
`knnMatch(k=2)`, and apply the ratio test.  Note that ORB uses Hamming distance.

In [ ]:
img_ex2_a = make_shapes_image(h=300, w=400)
M_ex2 = np.float32([[1, 0, 20], [0, 1, 15]])
img_ex2_b = cv2.warpAffine(img_ex2_a, M_ex2, (400, 300))

def orb_match_with_ratio_test(img_a, img_b, ratio_thresh=0.75):
    """
    Exercise 2: ORB detection + matching + Lowe's ratio test.

    Steps:
    1. Create ORB detector with cv2.ORB_create()
    2. Detect and compute features in both images
    3. Create BFMatcher with NORM_HAMMING
    4. Use knnMatch with k=2
    5. Apply ratio test: keep match if m.distance < ratio_thresh * n.distance

    Returns
    -------
    kp_a, kp_b : keypoints
    good_matches : list of good DMatch objects
    """
    orb = cv2.ORB_create(nfeatures=1000)

    kp_a, des_a = orb.detectAndCompute(img_a, None)
    kp_b, des_b = orb.detectAndCompute(img_b, None)

    if des_a is None or des_b is None or len(kp_a) < 2 or len(kp_b) < 2:
        return kp_a or [], kp_b or [], []

    bf = cv2.BFMatcher(cv2.NORM_HAMMING)
    matches = bf.knnMatch(des_a, des_b, k=2)

    # Ratio test: keep match if d₁/d₂ < ratio_thresh
    good_matches = []
    for m_pair in matches:
        if len(m_pair) == 2:
            m, n = m_pair
            if m.distance < ratio_thresh * n.distance:
                good_matches.append(m)

    assert len(good_matches) > 0, "Ratio test should retain matches on shifted views"
    return kp_a, kp_b, good_matches


kp_a_ex2, kp_b_ex2, matches_ex2 = orb_match_with_ratio_test(img_ex2_a, img_ex2_b)

img_ex2_vis = cv2.drawMatches(
    img_ex2_a, kp_a_ex2, img_ex2_b, kp_b_ex2, matches_ex2[:30], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)
plt.figure(figsize=(14, 5))
plt.imshow(img_ex2_vis)
plt.title(f"Exercise 2: ORB + ratio test ({len(matches_ex2)} matches)")
plt.axis("off")
plt.tight_layout()
plt.show()

---

### Exercise 3: RANSAC for Fundamental Matrix

Implement RANSAC with the 8-point algorithm to robustly estimate the fundamental matrix
from noisy correspondences with outliers.

In [ ]:
pts3d_ex3, K_ex3, R_ex3, t_ex3, pts1_ex3, pts2_ex3 = make_two_view_scene(
    n_points=60, noise_px=0.3
)

n_outliers_ex3 = 20
rng_ex3 = np.random.RandomState(123)
pts1_outliers = rng_ex3.rand(n_outliers_ex3, 2) * np.array([640, 480])
pts2_outliers = rng_ex3.rand(n_outliers_ex3, 2) * np.array([640, 480])

pts1_ex3_all = np.vstack([pts1_ex3, pts1_outliers])
pts2_ex3_all = np.vstack([pts2_ex3, pts2_outliers])
true_inlier_mask_ex3 = np.array([True] * len(pts1_ex3) + [False] * n_outliers_ex3)

print(f"Total points: {len(pts1_ex3_all)}")
print(f"True inliers: {true_inlier_mask_ex3.sum()}, outliers: {(~true_inlier_mask_ex3).sum()}")

In [ ]:
def ransac_fundamental(pts1, pts2, threshold=2.0, max_iter=1000, p=0.99):
    """
    Exercise 3: RANSAC + 8-point algorithm for F estimation.

    Steps:
    1. Randomly sample 8 correspondences
    2. Estimate F using eight_point_algorithm()
    3. Compute Sampson distance for all correspondences:
       d_i = (x2^T F x1)^2 / ((Fx1)_1^2 + (Fx1)_2^2 + (F^T x2)_1^2 + (F^T x2)_2^2)
    4. Count inliers (d_i < threshold)
    5. Keep best model, repeat

    Returns
    -------
    best_F : (3, 3) fundamental matrix
    inlier_mask : boolean array
    """
    N = len(pts1)
    best_F = None
    best_n_inliers = 0
    best_mask = np.zeros(N, dtype=bool)
    rng = np.random.RandomState(42)

    pts1_h = np.hstack([pts1, np.ones((N, 1))])
    pts2_h = np.hstack([pts2, np.ones((N, 1))])

    for it in range(max_iter):
        # Minimal sample (8 points for 8-point algorithm)
        idx = rng.choice(N, size=8, replace=False)

        # Estimate F from the sample
        try:
            F_cand = eight_point_algorithm(pts1[idx], pts2[idx])
        except Exception:
            continue

        # Sampson distance: algebraic approximation to geometric error
        Fx1 = (F_cand @ pts1_h.T).T          # (N, 3)
        Ftx2 = (F_cand.T @ pts2_h.T).T       # (N, 3)
        numerator = np.sum(pts2_h * Fx1, axis=1) ** 2
        denominator = (Fx1[:, 0] ** 2 + Fx1[:, 1] ** 2 +
                       Ftx2[:, 0] ** 2 + Ftx2[:, 1] ** 2)
        d = np.sqrt(numerator / np.maximum(denominator, 1e-12))

        # Count inliers
        mask = d < threshold
        n_inliers = mask.sum()
        if n_inliers > best_n_inliers:
            best_n_inliers = n_inliers
            best_F = F_cand
            best_mask = mask

    # Refit F on all inliers for improved accuracy
    if best_F is not None and best_n_inliers >= 8:
        best_F = eight_point_algorithm(pts1[best_mask], pts2[best_mask])
        Fx1 = (best_F @ pts1_h.T).T
        Ftx2 = (best_F.T @ pts2_h.T).T
        numerator = np.sum(pts2_h * Fx1, axis=1) ** 2
        denominator = (Fx1[:, 0] ** 2 + Fx1[:, 1] ** 2 +
                       Ftx2[:, 0] ** 2 + Ftx2[:, 1] ** 2)
        best_mask = np.sqrt(numerator / np.maximum(denominator, 1e-12)) < threshold

    assert best_F is not None, "RANSAC should find a valid fundamental matrix"
    assert best_mask.sum() >= 8, "Need at least 8 inliers"
    return best_F, best_mask


F_ex3, mask_ex3 = ransac_fundamental(pts1_ex3_all, pts2_ex3_all)

precision_ex3 = (mask_ex3 & true_inlier_mask_ex3).sum() / max(mask_ex3.sum(), 1)
recall_ex3 = (mask_ex3 & true_inlier_mask_ex3).sum() / true_inlier_mask_ex3.sum()
print(f"RANSAC found {mask_ex3.sum()} inliers")
print(f"Precision: {precision_ex3:.3f}, Recall: {recall_ex3:.3f}")
assert precision_ex3 > 0.8, f"RANSAC precision too low: {precision_ex3:.3f}"
assert recall_ex3 > 0.8, f"RANSAC recall too low: {recall_ex3:.3f}"

---

### Exercise 4: Draw Epipolar Lines

Given an estimated fundamental matrix $F$ and a set of point correspondences, draw the
epipolar lines in both images.  Verify that each point lies on (or near) its corresponding
epipolar line.

**Hint**: For a line $ax + by + c = 0$ and point $(x_0, y_0)$, the signed distance is:

$$
d = \frac{|a x_0 + b y_0 + c|}{\sqrt{a^2 + b^2}}
$$

In [ ]:
def draw_epipolar_lines_exercise(F, pts1, pts2, img_shape=(480, 640), n_lines=12):
    """
    Exercise 4: Draw epipolar lines.

    For each selected correspondence (x, x'):
    - The epipolar line in image 2 is: l' = F @ x   (line ax + by + c = 0)
    - The epipolar line in image 1 is: l  = F^T @ x'

    Steps:
    1. Select n_lines random correspondences
    2. For each, compute the epipolar line in the other image
    3. Plot the line by finding two endpoints at the image borders
    4. Plot the corresponding point in the same color
    5. Compute the point-to-line distance to verify accuracy
    """
    h, w = img_shape

    def line_endpoints(line, width, height):
        """Two border points for homogeneous line l = (a, b, c), ax + by + c = 0."""
        a, b, c = line
        pts = []
        if abs(b) > 1e-8:
            for x in [0, width - 1]:
                y = -(a * x + c) / b
                if 0 <= y < height:
                    pts.append((x, y))
        if abs(a) > 1e-8:
            for y in [0, height - 1]:
                x = -(b * y + c) / a
                if 0 <= x < width:
                    pts.append((x, y))
        return pts[:2]

    def point_line_distance(line, x, y):
        """Signed distance from (x, y) to line ax + by + c = 0."""
        a, b, c = line
        return abs(a * x + b * y + c) / np.sqrt(a ** 2 + b ** 2 + 1e-12)

    rng = np.random.RandomState(0)
    indices = rng.choice(len(pts1), min(n_lines, len(pts1)), replace=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    distances = []

    for idx in indices:
        x1, y1 = pts1[idx]
        x2, y2 = pts2[idx]
        color = np.random.rand(3,)

        # Epipolar line in image 2: l' = F @ x̃₁  (x' should lie on this line)
        l2 = F @ np.array([x1, y1, 1.0])
        endpoints = line_endpoints(l2, w, h)
        if len(endpoints) == 2:
            axes[1].plot([endpoints[0][0], endpoints[1][0]],
                         [endpoints[0][1], endpoints[1][1]], c=color, lw=1)
        axes[1].plot(x2, y2, 'o', color=color, markersize=6)
        distances.append(point_line_distance(l2, x2, y2))

        # Epipolar line in image 1: l = Fᵀ @ x̃₂
        l1 = F.T @ np.array([x2, y2, 1.0])
        endpoints = line_endpoints(l1, w, h)
        if len(endpoints) == 2:
            axes[0].plot([endpoints[0][0], endpoints[1][0]],
                         [endpoints[0][1], endpoints[1][1]], c=color, lw=1)
        axes[0].plot(x1, y1, 'o', color=color, markersize=6)
        distances.append(point_line_distance(l1, x1, y1))

    mean_dist = np.mean(distances)
    print(f"Mean point-to-epipolar-line distance: {mean_dist:.4f} px")
    assert mean_dist < 2.0, f"Points should lie near epipolar lines, got {mean_dist:.3f} px"

    axes[0].set_xlim(0, w)
    axes[0].set_ylim(h, 0)
    axes[0].set_title("Image 1: epipolar lines")
    axes[1].set_xlim(0, w)
    axes[1].set_ylim(h, 0)
    axes[1].set_title("Image 2: epipolar lines")
    plt.suptitle("Exercise 4: Epipolar Lines")
    plt.tight_layout()
    plt.show()


F_for_ex4 = eight_point_algorithm(pts1_2v, pts2_2v)
draw_epipolar_lines_exercise(F_for_ex4, pts1_2v, pts2_2v)

---

### Exercise 5: SuperPoint + LightGlue vs Classical Comparison

Run both classical (ORB) and learned (SuperPoint + LightGlue) matching on the same
image pair. Compare match count, inlier ratio after RANSAC, and visual quality.

In [ ]:
import cv2
import sys; sys.path.insert(0, "..")
from src.features import detect_orb, detect_sift, bf_match, ratio_test

def compare_matchers(img1_gray: np.ndarray, img2_gray: np.ndarray):
    """Compare ORB+BF vs SIFT+BF matching on an image pair.
    
    Parameters
    ----------
    img1_gray, img2_gray : (H, W) uint8 arrays
    
    Returns
    -------
    results : dict
        Keys: 'orb_matches', 'sift_matches', 'orb_time', 'sift_time'
    """
    import time
    
    # ORB + brute-force
    t0 = time.time()
    kp1_orb, des1_orb = detect_orb(img1_gray, n_features=500)
    kp2_orb, des2_orb = detect_orb(img2_gray, n_features=500)
    matches_orb = bf_match(des1_orb, des2_orb, norm_type=cv2.NORM_HAMMING)
    t_orb = time.time() - t0
    
    # SIFT + brute-force + ratio test
    t0 = time.time()
    kp1_sift, des1_sift = detect_sift(img1_gray, n_features=500)
    kp2_sift, des2_sift = detect_sift(img2_gray, n_features=500)
    matches_sift = bf_match(des1_sift, des2_sift, norm_type=cv2.NORM_L2)
    good_sift = ratio_test(matches_sift, ratio=0.75) if len(matches_sift) > 0 else []
    t_sift = time.time() - t0
    
    return {
        'orb_matches': len(matches_orb),
        'sift_matches': len(good_sift),
        'orb_time_ms': t_orb * 1000,
        'sift_time_ms': t_sift * 1000,
    }

np.random.seed(42)
img1 = np.random.randint(50, 200, (240, 320), dtype=np.uint8)
img1 = cv2.GaussianBlur(img1, (5, 5), 2.0)
M = cv2.getRotationMatrix2D((160, 120), 5, 1.0)
img2 = cv2.warpAffine(img1, M, (320, 240))

results = compare_matchers(img1, img2)
print(f"ORB matches: {results['orb_matches']} in {results['orb_time_ms']:.1f} ms")
print(f"SIFT matches: {results['sift_matches']} in {results['sift_time_ms']:.1f} ms")

assert results['orb_matches'] > 0, "ORB should find some matches"
assert results['sift_matches'] > 0, "SIFT should find some matches"

---

## Summary

This notebook covered the complete pipeline from **feature detection** to **two-view
geometry estimation**:

| Stage | Classical | Learned |
|:---|:---|:---|
| **Detection** | Harris, FAST | SuperPoint |
| **Description** | SIFT (128-D float), ORB (256-bit binary) | SuperPoint desc., DINOv2 |
| **Matching** | BFMatcher, FLANN + ratio test | LightGlue, LoFTR, RoMa |
| **Robust estimation** | RANSAC | RANSAC (still needed!) |
| **Geometry** | Fundamental/Essential matrix, $(R,t)$ recovery | Same |

**Key mathematical results**:

1. **Harris response**: $R = \lambda_1\lambda_2 - k(\lambda_1+\lambda_2)^2$ discriminates
   corners from edges and flat regions.

2. **RANSAC iterations**: $N = \frac{\log(1-p)}{\log(1-(1-\varepsilon)^s)}$ bounds
   the number of trials needed for robust estimation.

3. **Epipolar constraint**: $\mathbf{x}'^\top F\mathbf{x} = 0$ encodes the geometric
   relationship between two views.

4. **Essential matrix decomposition**: $E \to (R, \mathbf{t})$ with cheirality check
   recovers camera motion up to scale.

**Next notebook**: Stereo vision, disparity estimation, and dense reconstruction.